# Tablero de Calidad de Datos — Primas (solo lectura)

Este notebook es **solo de lectura**: ejecuta únicamente `SELECT` contra Redshift (nunca `CREATE`/`INSERT`/`DELETE`/`CALL`). No depende de que el cubo `co_sandbox_datos.kpi_cubo_mensual` exista ni esté poblado — todo se calcula al vuelo desde la vista fuente.

Qué hace:

1. Se conecta a Redshift.
2. **Detecta automáticamente** los últimos periodos contables cerrados **y el mes en curso (con datos parciales)** en la fuente (`gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement`) — sin periodos quemados, a diferencia de los `sql/01..05` originales.
3. Calcula los 5 indicadores (Completitud, Exactitud, Unicidad, Validez, Disponibilidad) con `SELECT` directos sobre la fuente, usando las mismas reglas de negocio de los archivos originales. Única desviación deliberada: aquí Exactitud **sí** aplica el filtro compartido completo (incluye `current_record_flag = 1`) — ver hallazgo #1 en [`docs/Hallazgos y Estado del Proyecto.md`](../docs/Hallazgos%20y%20Estado%20del%20Proyecto.md).
4. Genera un **tablero HTML autocontenido** (sin dependencias externas) en `reports/tablero_calidad_primas.html` y lo abre en el navegador.

In [1]:
import json
import webbrowser
from datetime import datetime
from pathlib import Path

import pandas as pd
import redshift_connector

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

## Conexion a Redshift
Ajusta host/puerto/base si hace falta; usuario y contraseña se piden de forma interactiva.

In [2]:
import getpass

In [3]:
# Credenciales fuera del repo: se cargan desde notebooks/credenciales_local.py,
# que está en .gitignore (así no se exponen y tampoco hay que digitarlas cada vez).
# Si el archivo no existe, se piden por consola sin quedar guardadas en ninguna parte.
try:
    from credenciales_local import (
        DEFAULT_HOST, DEFAULT_PORT, DEFAULT_DATABASE, DEFAULT_USER, DEFAULT_PASSWORD
    )
    print('Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).')
except ImportError:
    print('No se encontró credenciales_local.py — ingresa las credenciales:')
    DEFAULT_HOST = 'corshftanltc-dprogramp.hdicolombia.com.co'
    DEFAULT_PORT = '9519'
    DEFAULT_DATABASE = 'adp_dwh'
    DEFAULT_USER = input('Usuario Redshift: ')
    DEFAULT_PASSWORD = getpass.getpass('Contraseña Redshift: ')

Credenciales cargadas desde credenciales_local.py (archivo local, fuera de git).


In [4]:
host = DEFAULT_HOST
port = DEFAULT_PORT
database =  DEFAULT_DATABASE
user = DEFAULT_USER
password = DEFAULT_PASSWORD

In [5]:
conn = redshift_connector.connect(
    host=host,
    port=int(port),
    database=database,
    user=user,
    password=password,
)
cursor = conn.cursor()
print('Conectado (solo lectura).')


Conectado (solo lectura).


In [6]:
def run_query(sql):
    """Ejecuta un SELECT (solo lectura) y devuelve un DataFrame."""
    cursor.execute(sql)
    return cursor.fetch_dataframe()


## 0) Fuentes de datos: ¿sobre qué se hace el DQ?

Todo el data quality de este notebook se calcula sobre **una sola fuente principal**:

| Objeto | Tipo | Rol |
|---|---|---|
| `gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement` | Vista | Hechos de primas. **Todos** los indicadores (Completitud, Exactitud, Unicidad, Validez y Disponibilidad regla 4) leen de aquí. |
| `gde_adp_dwh.fact_policy_transaction_movement` | Tabla base | Solo la usa **Disponibilidad (sync)** para comparar conteos tabla vs. vista (rezago de refresco). |

Antes de calcular nada, la celda siguiente trae **10 filas de ejemplo** de la vista
para familiarizarse con las columnas y los valores reales.


In [7]:
# Muestra de 10 filas de la vista fuente, para entender la base antes del DQ.
import pandas as pd

FUENTE_VISTA = 'gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement'
FUENTE_TABLA = 'gde_adp_dwh.fact_policy_transaction_movement'

print(f'Fuente principal (todos los KPIs) : {FUENTE_VISTA}')
print(f'Fuente secundaria (solo Disponibilidad sync): {FUENTE_TABLA}')

df_muestra = run_query(f'''
    select *
    from {FUENTE_VISTA}
    where current_record_flag = 1
    limit 10
''')

print(f'\nColumnas disponibles ({len(df_muestra.columns)}):')
print(list(df_muestra.columns))

# Mostrar las 10 filas completas (sin truncar columnas)
with pd.option_context('display.max_columns', None, 'display.width', None):
    display(df_muestra)


Fuente principal (todos los KPIs) : gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
Fuente secundaria (solo Disponibilidad sync): gde_adp_dwh.fact_policy_transaction_movement



Columnas disponibles (65):
['policy_transaction_movement_sk', 'source_system', 'audit_transaction_id', 'customer_sk', 'quote_sk', 'policy_sk', 'risk_sk', 'policy_item_sk', 'insured_sk', 'vehicle_sk', 'coverage_sk', 'producer_sk', 'branch_sk', 'product_sk', 'lob_sk', 'risk_id', 'sseguro', 'risk_number', 'policy_number', 'coverage_code', 'product_code', 'transaction_date_sk', 'policy_effective_date_sk', 'policy_expiration_date_sk', 'inception_date_sk', 'transaction_type', 'transaction_type_description', 'movement_type', 'transaction_effective_date_sk', 'transaction_expiration_date_sk', 'receipt_type', 'receipt_number', 'transaction_delta_billed_premium_amount', 'transaction_delta_billed_premium_amount_raw', 'transaction_delta_net_written_premium_amount', 'transaction_delta_net_written_premium_amount_raw', 'transaction_delta_base_premium_amount', 'transaction_delta_base_annual_premium_amount', 'transaction_delta_commission_amount', 'transaction_delta_commission_amount_raw', 'transaction_

,policy_transaction_movement_sk,source_system,audit_transaction_id,customer_sk,quote_sk,policy_sk,risk_sk,policy_item_sk,insured_sk,vehicle_sk,coverage_sk,producer_sk,branch_sk,product_sk,lob_sk,risk_id,sseguro,risk_number,policy_number,coverage_code,product_code,transaction_date_sk,policy_effective_date_sk,policy_expiration_date_sk,inception_date_sk,transaction_type,transaction_type_description,movement_type,transaction_effective_date_sk,transaction_expiration_date_sk,receipt_type,receipt_number,transaction_delta_billed_premium_amount,transaction_delta_billed_premium_amount_raw,transaction_delta_net_written_premium_amount,transaction_delta_net_written_premium_amount_raw,transaction_delta_base_premium_amount,transaction_delta_base_annual_premium_amount,transaction_delta_commission_amount,transaction_delta_commission_amount_raw,transaction_delta_total_tax_amount,transaction_delta_concierge_tax_amount,transaction_delta_clea_tax_amount,transaction_delta_fng_tax_amount,transaction_delta_ips_tax_amount,transaction_delta_ie_tax_amount,transaction_delta_ipt_tax_amount,transaction_delta_miicf_tax_amount,transaction_delta_compens_fund_tax_amount,transaction_delta_statutory_tax_amount,transaction_delta_statutory_ie_tax_amount,transaction_accounting_ts,transaction_delta_consorcio_occupant_tax_amount,transaction_delta_selo_tax_amount,transaction_delta_selo_dac_tax_amount,load_ts,transaction_delta_uninsured_auto_fund_fee_amount,transaction_delta_road_safety_fee_amount,transaction_delta_medical_emergency_fund_fee_amount,transaction_delta_green_card_fee_amount,performer_login,creator_login,current_record_flag,accountable_period,annual_premium
0,13254312072,CO_iaxis,10459377-1-766-25601142,857959943,-1,484163957,964842328,13237299,367458292,60264910,6838943782,136902,461,1175,543,8521234-1,8521234,0,241738,766,6031,20180620,20180619,20190619,20180620,0,Nueva producción,None,20180619,-1,not-unificado,None,41696.0,41696.0,41696.0,41696.00,None,None,5212.0,5212.0,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,41696.0000000000
1,13254320038,CO_iaxis,10490341-1-760-25656193,857959943,-1,484163956,964842329,13237299,367458292,60264910,6838943776,136902,461,1175,543,8521234-2,8521234,0,241738,760,6031,20180622,20180619,20190619,20180620,3,Anulación,None,20180619,-1,not-unificado,None,-378391.0,-378391.0,-378391.0,-378391.00,None,None,-47299.0,-47299.0,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,None
2,13432766045,CO_iaxis,10504034-1-766-25675513,856501675,-1,484180976,964871080,13246123,362189608,60270948,6839069217,136902,461,1175,543,8554098-1,8554098,0,242396,766,6031,20180625,20180623,20190623,20180625,0,Nueva producción,None,20180623,-1,not-unificado,None,41696.0,41696.0,41696.0,41696.00,None,None,5212.0,5212.0,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,KATERIN.MATEUSL,1,200806,41696.0000000000
3,13254320040,CO_iaxis,10490341-1-766-25656193,857959943,-1,484163956,964842329,13237299,367458292,60264910,6838943782,136902,461,1175,543,8521234-2,8521234,0,241738,766,6031,20180622,20180619,20190619,20180620,3,Anulación,None,20180619,-1,not-unificado,None,-41696.0,-41696.0,-41696.0,-41696.00,None,None,-5212.0,-5212.0,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,None
4,13313783252,CO_iaxis,10459377-1-773-25601142,857959943,-1,484163957,964842328,13237299,367458292,60264910,6838943787,136902,461,1175,543,8521234-1,8521234,0,241738,773,6031,20180620,20180619,20190619,20180620,0,Nueva producción,None,20180619,-1,not-unificado,None,11789.0,11789.0,11789.0,11789.00,None,None,1474.0,1474.0,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,N

In [8]:
df_muestra.head()

,policy_transaction_movement_sk,source_system,audit_transaction_id,customer_sk,quote_sk,policy_sk,risk_sk,policy_item_sk,insured_sk,vehicle_sk,coverage_sk,producer_sk,branch_sk,product_sk,lob_sk,risk_id,sseguro,risk_number,policy_number,coverage_code,product_code,transaction_date_sk,policy_effective_date_sk,policy_expiration_date_sk,inception_date_sk,...,transaction_delta_total_tax_amount,transaction_delta_concierge_tax_amount,transaction_delta_clea_tax_amount,transaction_delta_fng_tax_amount,transaction_delta_ips_tax_amount,transaction_delta_ie_tax_amount,transaction_delta_ipt_tax_amount,transaction_delta_miicf_tax_amount,transaction_delta_compens_fund_tax_amount,transaction_delta_statutory_tax_amount,transaction_delta_statutory_ie_tax_amount,transaction_accounting_ts,transaction_delta_consorcio_occupant_tax_amount,transaction_delta_selo_tax_amount,transaction_delta_selo_dac_tax_amount,load_ts,transaction_delta_uninsured_auto_fund_fee_amount,transaction_delta_road_safety_fee_amount,transaction_delta_medical_emergency_fund_fee_amount,transaction_delta_green_card_fee_amount,performer_login,creator_login,current_record_flag,accountable_period,annual_premium
0,13254312072,CO_iaxis,10459377-1-766-25601142,857959943,-1,484163957,964842328,13237299,367458292,60264910,6838943782,136902,461,1175,543,8521234-1,8521234,0,241738,766,6031,20180620,20180619,20190619,20180620,...,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,41696.0000000000
1,13254320038,CO_iaxis,10490341-1-760-25656193,857959943,-1,484163956,964842329,13237299,367458292,60264910,6838943776,136902,461,1175,543,8521234-2,8521234,0,241738,760,6031,20180622,20180619,20190619,20180620,...,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,None
2,13432766045,CO_iaxis,10504034-1-766-25675513,856501675,-1,484180976,964871080,13246123,362189608,60270948,6839069217,136902,461,1175,543,8554098-1,8554098,0,242396,766,6031,20180625,20180623,20190623,20180625,...,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,KATERIN.MATEUSL,1,200806,41696.0000000000
3,13254320040,CO_iaxis,10490341-1-766-25656193,857959943,-1,484163956,964842329,13237299,367458292,60264910,6838943782,136902,461,1175,543,8521234-2,8521234,0,241738,766,6031,20180622,20180619,20190619,20180620,...,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,None
4,13313783252,CO_iaxis,10459377-1-773-25601142,857959943,-1,484163957,964842328,13237299,367458292,60264910,6838943787,136902,461,1175,543,8521234-1,8521234,0,241738,773,6031,20180620,20180619,20190619,20180620,...,None,None,None,None,None,None,None,None,None,0,None,2008-06-20,None,None,None,2026-06-09 20:28:27.882443,None,None,None,None,None,ANDREA.PARADAZ,1,200806,11789.0000000000


In [9]:
df_muestra.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 65 columns):
 #   Column                                               Non-Null Count  Dtype         
---  ------                                               --------------  -----         
 0   policy_transaction_movement_sk                       10 non-null     int64         
 1   source_system                                        10 non-null     str           
 2   audit_transaction_id                                 10 non-null     str           
 3   customer_sk                                          10 non-null     int64         
 4   quote_sk                                             10 non-null     int64         
 5   policy_sk                                            10 non-null     int64         
 6   risk_sk                                              10 non-null     int64         
 7   policy_item_sk                                       10 non-null     int64         
 8   insured_sk    

### Diccionario de campos de la fuente

La tabla siguiente documenta **cada columna** de la vista: qué significa, su tipo, un valor
de ejemplo real (tomado de `df_muestra`) y en qué **reglas de DQ** participa (más si entra
al filtro base del universo). Las definiciones viven en `DICCIONARIO_CAMPOS` y también se
incrustan en el tablero HTML para el drill-down por campo.

> Los impuestos/tasas poco frecuentes llevan una definición genérica — afinar con negocio
> cuando estén documentados oficialmente.


In [10]:
# Diccionario de campos de la vista fuente. Las definiciones se reutilizan en el
# tablero HTML (drill-down al hacer clic en un anillo del resumen).
DICCIONARIO_CAMPOS = {
    # ---- Identificadores / llaves surrogate (las *_sk apuntan a dimensiones; -1 = no aplica / no informado) ----
    'policy_transaction_movement_sk': 'Llave única (surrogate) del movimiento de prima. Es la llave que Unicidad valida contra duplicados.',
    'source_system': "Sistema origen del registro: 'CO_iaxis' (iAxis) o 'CO_as400' (AS/400).",
    'audit_transaction_id': 'Identificador de auditoría de la transacción en el sistema origen (trazabilidad).',
    'customer_sk': 'Llave surrogate del cliente (tomador).',
    'quote_sk': 'Llave surrogate de la cotización; -1 = no aplica / no informado.',
    'policy_sk': 'Llave surrogate de la póliza.',
    'risk_sk': 'Llave surrogate del riesgo asegurado.',
    'policy_item_sk': 'Llave surrogate del ítem/objeto asegurado dentro de la póliza.',
    'insured_sk': 'Llave surrogate del asegurado.',
    'vehicle_sk': 'Llave surrogate del vehículo (ramos de autos); -1 = no aplica.',
    'coverage_sk': 'Llave surrogate de la cobertura.',
    'producer_sk': 'Llave surrogate del intermediario / productor.',
    'branch_sk': 'Llave surrogate de la sucursal.',
    'product_sk': 'Llave surrogate del producto.',
    'lob_sk': 'Llave surrogate de la línea de negocio (Line of Business).',
    # ---- Identificadores de negocio ----
    'risk_id': 'Identificador del riesgo en el sistema origen.',
    'sseguro': 'Identificador interno del seguro en iAxis. CO_as400 no lo puebla (excepción por fuente en Completitud).',
    'risk_number': 'Número del riesgo dentro de la póliza.',
    'policy_number': 'Número de la póliza.',
    'coverage_code': 'Código de la cobertura. El código 8888 se excluye del universo DQ (filtro base).',
    'product_code': 'Código del producto/ramo. En CO_iaxis debe ser numérico; letras solo se aceptan en CO_as400 (regla de Validez). Es el "ramo" de Disponibilidad regla 4.',
    # ---- Fechas como enteros YYYYMMDD / YYYYMM ----
    'transaction_date_sk': 'Fecha de la transacción, entero con forma YYYYMMDD.',
    'policy_effective_date_sk': 'Inicio de vigencia de la póliza (YYYYMMDD).',
    'policy_expiration_date_sk': 'Fin de vigencia de la póliza (YYYYMMDD).',
    'inception_date_sk': 'Fecha de inicio original (incepción) de la relación/póliza (YYYYMMDD).',
    'transaction_effective_date_sk': 'Inicio de vigencia del movimiento (YYYYMMDD).',
    'transaction_expiration_date_sk': 'Fin de vigencia del movimiento (YYYYMMDD).',
    'accountable_period': 'Periodo contable del movimiento, entero YYYYMM (p. ej. 202606). Es el periodo_contable por el que se cortan todos los KPIs.',
    # ---- Transacción / movimiento ----
    'transaction_type': 'Código del tipo de transacción (emisión, endoso, cancelación, …).',
    'transaction_type_description': 'Descripción legible del tipo de transacción.',
    'movement_type': 'Tipo de movimiento contable. Excluido de Validez a propósito (igual que en el SQL original).',
    'receipt_type': "Tipo de recibo: 'not-unificado', 'unificado-detail' o 'unificado-total'. Los 'unificado-total' de iAxis se excluyen del universo para no duplicar prima.",
    'receipt_number': 'Número del recibo. Completitud le aplica reglas por fuente (CO_as400 y los not-unificado no lo llevan).',
    'transaction_accounting_ts': 'Fecha/hora contable de la transacción (timestamp).',
    'load_ts': 'Fecha/hora en que el registro se cargó al DWH.',
    'performer_login': 'Usuario/proceso que ejecutó la transacción en el origen.',
    'creator_login': 'Usuario/proceso que creó el registro en el origen.',
    'current_record_flag': 'Bandera de vigencia (SCD): 1 = versión vigente del registro, 0 = histórica. Todo el DQ filtra por 1.',
    # ---- Montos de prima ("delta" = lo que aporta ESTE movimiento; "_raw" = valor crudo del origen) ----
    'transaction_delta_billed_premium_amount': 'Delta de prima facturada del movimiento. El universo DQ exige que sea ≠ 0 (filtro base).',
    'transaction_delta_billed_premium_amount_raw': 'Prima facturada del movimiento sin transformar, tal como llegó del origen.',
    'transaction_delta_net_written_premium_amount': 'Delta de prima neta emitida.',
    'transaction_delta_net_written_premium_amount_raw': 'Prima neta emitida sin transformar (valor crudo del origen).',
    'transaction_delta_base_premium_amount': 'Delta de prima base (sin impuestos ni recargos).',
    'transaction_delta_base_annual_premium_amount': 'Delta de prima base anualizada.',
    'transaction_delta_commission_amount': 'Delta de comisión del intermediario.',
    'transaction_delta_commission_amount_raw': 'Comisión sin transformar (valor crudo del origen).',
    'annual_premium': 'Prima anual asociada a la póliza/riesgo.',
    # ---- Impuestos y contribuciones (delta del movimiento; componentes específicos por país/producto —
    #      definición genérica, afinar con negocio cuando se documenten oficialmente) ----
    'transaction_delta_total_tax_amount': 'Delta del total de impuestos del movimiento (suma de los componentes).',
    'transaction_delta_concierge_tax_amount': 'Delta de impuesto/tasa "concierge" (componente específico por país/producto).',
    'transaction_delta_clea_tax_amount': 'Delta de impuesto/tasa CLEA (componente específico por país/producto).',
    'transaction_delta_fng_tax_amount': 'Delta de contribución FNG (componente específico por país/producto).',
    'transaction_delta_ips_tax_amount': 'Delta de impuesto/tasa IPS (componente específico por país/producto).',
    'transaction_delta_ie_tax_amount': 'Delta de impuesto/tasa IE (componente específico por país/producto).',
    'transaction_delta_ipt_tax_amount': 'Delta de impuesto sobre la prima (IPT — Insurance Premium Tax).',
    'transaction_delta_miicf_tax_amount': 'Delta de impuesto/tasa MIICF (componente específico por país/producto).',
    'transaction_delta_compens_fund_tax_amount': 'Delta de contribución a fondo de compensación.',
    'transaction_delta_statutory_tax_amount': 'Delta de impuesto/contribución estatutaria.',
    'transaction_delta_statutory_ie_tax_amount': 'Delta de impuesto/contribución estatutaria IE.',
    'transaction_delta_consorcio_occupant_tax_amount': 'Delta de tasa de consorcio / ocupantes (componente específico por país/producto).',
    'transaction_delta_selo_tax_amount': 'Delta de impuesto de estampilla ("selo").',
    'transaction_delta_selo_dac_tax_amount': 'Delta de impuesto de estampilla DAC ("selo DAC").',
    # ---- Tasas / fees ----
    'transaction_delta_uninsured_auto_fund_fee_amount': 'Delta de contribución al fondo de vehículos no asegurados.',
    'transaction_delta_road_safety_fee_amount': 'Delta de tasa de seguridad vial.',
    'transaction_delta_medical_emergency_fund_fee_amount': 'Delta de contribución al fondo de emergencias médicas.',
    'transaction_delta_green_card_fee_amount': 'Delta de tasa de carta verde (cobertura fronteriza).',
}


# Las definiciones internas se complementan/sobrescriben con el diccionario oficial
# exportado de la vista: notebooks/vw_fact_policy_transaction_movement.csv
# (columnas: Campo, Descripción, Tipo de dato, Ejemplo). Esta misma definición es
# la que ve el usuario del tablero HTML al pasar el mouse sobre un campo.
_CSV_DICC = Path('vw_fact_policy_transaction_movement.csv')
if not _CSV_DICC.exists():
    _CSV_DICC = Path('notebooks') / 'vw_fact_policy_transaction_movement.csv'
if _CSV_DICC.exists():
    _df_dicc_csv = pd.read_csv(_CSV_DICC)
    _n_csv = 0
    for _r in _df_dicc_csv.itertuples(index=False):
        _campo = str(_r[0]).strip()
        _desc = '' if pd.isna(_r[1]) else str(_r[1]).strip()
        if not _campo or not _desc:
            continue
        _tipo = '' if len(_r) < 3 or pd.isna(_r[2]) else str(_r[2]).replace('\n', ',').strip()
        # Repara tipos que quedaron partidos en el CSV, p. ej. "10),NUMERIC(16" -> "NUMERIC(16,10)"
        if _tipo[:1].isdigit() and ',' in _tipo:
            _a, _b = _tipo.split(',', 1)
            _tipo = _b + ',' + _a
        _ej = '' if len(_r) < 4 or pd.isna(_r[3]) else str(_r[3]).strip()
        _extras = ' · '.join(x for x in ('Tipo: ' + _tipo if _tipo else '', 'Ej: ' + _ej if _ej else '') if x)
        DICCIONARIO_CAMPOS[_campo] = _desc + (' (' + _extras + ')' if _extras else '')
        _n_csv += 1
    print(f'Definiciones cargadas desde {_CSV_DICC.name}: {_n_csv} campos.')
else:
    print('⚠ No se encontró vw_fact_policy_transaction_movement.csv; se usan solo las definiciones internas.')

# En qué reglas DQ participa cada campo — espejo estático de la sección 2
# (si cambias las reglas allá, actualiza esta copia).
USO_DQ = {
    'completitud': [
        'source_system', 'accountable_period', 'coverage_code', 'branch_sk', 'product_code',
        'transaction_date_sk', 'policy_effective_date_sk', 'policy_expiration_date_sk',
        'inception_date_sk', 'transaction_type', 'transaction_effective_date_sk',
        'transaction_type_description', 'current_record_flag',
        'transaction_delta_billed_premium_amount', 'transaction_delta_commission_amount',
        'risk_number', 'policy_number', 'transaction_delta_billed_premium_amount_raw',
        'policy_transaction_movement_sk', 'sseguro', 'receipt_type', 'receipt_number',
    ],
    'exactitud': ['source_system', 'current_record_flag', 'receipt_type'],
    'validez': [
        'source_system', 'coverage_code', 'product_code', 'transaction_date_sk',
        'transaction_type', 'current_record_flag', 'risk_number', 'policy_number',
        'accountable_period', 'policy_effective_date_sk', 'inception_date_sk',
        'transaction_effective_date_sk',
    ],
    'unicidad': ['policy_transaction_movement_sk'],
    'disponibilidad (regla 4)': ['product_code', 'transaction_date_sk'],
}
EN_FILTRO_BASE = ['accountable_period', 'current_record_flag', 'coverage_code',
                  'transaction_delta_billed_premium_amount', 'source_system', 'receipt_type']


def _ejemplo(col):
    s = df_muestra[col].dropna()
    return s.iloc[0] if len(s) else None


df_diccionario = pd.DataFrame([{
    'campo': c,
    'tipo_dato': str(df_muestra[c].dtype),
    'definicion': DICCIONARIO_CAMPOS.get(c, '(pendiente de definir)'),
    'ejemplo': _ejemplo(c),
    'reglas_dq': ', '.join(k for k, v in USO_DQ.items() if c in v) or '—',
    'filtro_base': 'sí' if c in EN_FILTRO_BASE else '',
} for c in df_muestra.columns])

faltan = [c for c in df_muestra.columns if c not in DICCIONARIO_CAMPOS]
if faltan:
    print('⚠ Columnas de la vista sin definición en el diccionario:', faltan)

print(f'{len(df_diccionario)} campos documentados.')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_diccionario)


Definiciones cargadas desde vw_fact_policy_transaction_movement.csv: 64 campos.


65 campos documentados.


,campo,tipo_dato,definicion,ejemplo,reglas_dq,filtro_base
0,policy_transaction_movement_sk,int64,Identificador único (surrogate key) del movimiento de transacción de póliza. (Tipo: BIGINT · Ej: 9794069171),13254312072,"completitud, unicidad",
1,source_system,str,"Sistema fuente del dato (ej. CO_iaxis, CO_as400). (Tipo: VARCHAR(50) · Ej: CO_iaxis)",CO_iaxis,"completitud, exactitud, validez",sí
2,audit_transaction_id,str,Identificador de auditoría de la transacción en procesos ETL/CDP. (Tipo: VARCHAR(50) · Ej: 73890630-1-9037-248275060),10459377-1-766-25601142,—,
3,customer_sk,int64,Clave surrogate del cliente para enlazar con la dimensión de clientes. (Tipo: INTEGER · Ej: 857634872),857959943,—,
4,quote_sk,int64,Clave surrogate de la cotización para enlazar con la dimensión de cotizaciones. (Tipo: INTEGER · Ej: -1),-1,—,
5,policy_sk,int64,Clave surrogate de la póliza para enlazar con la dimensión de pólizas. (Tipo: INTEGER · Ej: 488167784),484163957,—,
6,risk_sk,int64,Clave surrogate del riesgo para enlazar con la dimensión de riesgos. (Tipo: INTEGER · Ej: 952647505),964842328,—,
7,policy_item_sk,int64,Clave surrogate del ítem de póliza (objeto asegurado/cobertura específica). (Tipo: INTEGER · Ej: 9123182),13237299,—,
8,insured_sk,int64,Clave surrogate del asegurado (persona/natural o jurídica). (Tipo: INTEGER · Ej: 363671399),367458292,—,
9,vehicle_sk,int64,Clave surrogate del vehículo (aplica para ramo autos). (Tipo: INTEGER · Ej: 59866927),60264910,—,


## 1) Detección automática de periodos (sin fechas quemadas)

Se listan los periodos disponibles en la fuente y se toman los últimos `N_PERIODOS` **cerrados** (menores al mes actual) **más el mes en curso** si ya tiene datos (`INCLUIR_MES_EN_CURSO = True`). El mes en curso entra con cifras **parciales**: cada vez que se ejecute el notebook (hoy, mañana, etc.) se recalculan y el tablero muestra cómo van evolucionando. `disponibilidad_regla4` solo le exige los días ya transcurridos, para no castigar días que aún no ocurren. Si necesitas forzar periodos puntuales usa `PERIODOS_MANUAL`; si prefieres solo meses cerrados, pon `INCLUIR_MES_EN_CURSO = False`.

In [11]:
N_PERIODOS = 12              # cuántos periodos cerrados incluir en el tablero
INCLUIR_MES_EN_CURSO = True # True: agrega el mes actual (datos PARCIALES) si ya hay datos en la fuente
PERIODOS_MANUAL = None      # ej. [202601, 202602] para forzar periodos puntuales

df_periodos = run_query('''
select distinct accountable_period as periodo
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where current_record_flag = 1
order by 1;
''')

periodo_mes_actual = int(datetime.now().strftime('%Y%m'))
periodos_fuente = sorted(int(p) for p in df_periodos['periodo'])
periodos_cerrados = [p for p in periodos_fuente if p < periodo_mes_actual]

if PERIODOS_MANUAL:
    PERIODOS = sorted(int(p) for p in PERIODOS_MANUAL)
else:
    PERIODOS = periodos_cerrados[-N_PERIODOS:]
    if INCLUIR_MES_EN_CURSO and periodo_mes_actual in periodos_fuente:
        PERIODOS = PERIODOS + [periodo_mes_actual]

if not PERIODOS:
    raise ValueError('No se encontraron periodos en la fuente.')
periodos_sql = ', '.join(str(p) for p in PERIODOS)

# Mes en curso incluido (o None si no aplica): sus cifras son PARCIALES y evolucionan
# con cada corrida. disponibilidad_regla4 solo le exige los días ya transcurridos
# (hasta ayer, máx. 15) para no castigar días que aún no ocurren.
PERIODO_EN_CURSO = periodo_mes_actual if periodo_mes_actual in PERIODOS else None
DIAS_REGLA4_EN_CURSO = max(0, min(15, datetime.now().day - 1))

if PERIODO_EN_CURSO:
    print(f'Mes en curso incluido con datos parciales: {PERIODO_EN_CURSO} '
          f'(regla4 le exige días 1-{DIAS_REGLA4_EN_CURSO})')
else:
    print(f'Mes actual {periodo_mes_actual}: sin datos en la fuente todavía (o excluido).')
print(f'Periodos del tablero: {PERIODOS}')

Mes en curso incluido con datos parciales: 202607 (regla4 le exige días 1-14)
Periodos del tablero: [202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604, 202605, 202606, 202607]


## 2) Cálculo de los 5 indicadores (solo `SELECT`)

Cada bloque ejecuta **una sola consulta agregada** para todos los periodos elegidos y deja el resultado en formato largo (`tipo_indicador` + `nombre_campo` + `periodo_contable`), el mismo grano del cubo unificado.

- Completitud, Exactitud, Unicidad y Validez usan el **filtro base compartido** documentado en `CLAUDE.md`.
- Disponibilidad **no** lo usa a propósito (compara tabla base vs. vista completas para no esconder brechas de sincronización).

In [12]:
# Filtro base compartido (Completitud / Exactitud / Unicidad / Validez) — ver CLAUDE.md.
FILTRO_BASE = f'''accountable_period IN ({periodos_sql})
  AND current_record_flag = 1
  AND coverage_code <> 8888
  AND transaction_delta_billed_premium_amount <> 0
  AND (
        (source_system = 'CO_iaxis' AND receipt_type NOT IN ('unificado-total'))
        OR source_system = 'CO_as400'
      )'''


def a_formato_largo(df_ancho, tipo_indicador):
    """Convierte un DF con una columna por campo (conteo de filas malas) a formato largo."""
    df = df_ancho.melt(
        id_vars=['periodo_contable', 'total_registros'],
        var_name='nombre_campo', value_name='cantidad_mala',
    )
    df['periodo_contable'] = df['periodo_contable'].astype('int64')
    df['total_registros'] = df['total_registros'].astype('int64')
    df['cantidad_mala'] = df['cantidad_mala'].astype('int64')
    df['tipo_indicador'] = tipo_indicador
    df['porcentaje'] = (100.0 * (1 - df['cantidad_mala'] / df['total_registros'])).round(2)
    df['es_total'] = 0
    return df


# ---- COMPLETITUD: % de no-nulos por campo (19 obligatorios + 3 con excepciones por fuente)
CAMPOS_OBLIGATORIOS = [
    'source_system', 'accountable_period', 'coverage_code', 'branch_sk', 'product_code',
    'transaction_date_sk', 'policy_effective_date_sk', 'policy_expiration_date_sk',
    'inception_date_sk', 'transaction_type', 'transaction_effective_date_sk',
    'transaction_type_description', 'current_record_flag',
    'transaction_delta_billed_premium_amount', 'transaction_delta_commission_amount',
    'risk_number', 'policy_number', 'transaction_delta_billed_premium_amount_raw',
    'policy_transaction_movement_sk',
]
nulls_simples = ',\n    '.join(
    f'SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}' for c in CAMPOS_OBLIGATORIOS
)

sql_completitud = f'''
select
    accountable_period as periodo_contable,
    count(*) as total_registros,
    {nulls_simples},
    -- Campos con excepciones por fuente (CO_as400 no los puebla):
    SUM(CASE WHEN sseguro IS NULL AND source_system <> 'CO_as400' THEN 1 ELSE 0 END) AS sseguro,
    SUM(CASE WHEN receipt_type IS NULL AND source_system <> 'CO_as400' THEN 1 ELSE 0 END) AS receipt_type,
    SUM(CASE
            WHEN (receipt_number IS NULL AND source_system <> 'CO_as400' AND receipt_type <> 'not-unificado')
              OR (receipt_type = 'not-unificado' AND receipt_number IS NOT NULL)
            THEN 1 ELSE 0
        END) AS receipt_number
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where {FILTRO_BASE}
group by 1
order by 1;
'''

df_completitud = a_formato_largo(run_query(sql_completitud), 'completitud')
print(f'{len(df_completitud)} filas (campo x periodo). Peores campos del último periodo:')
df_completitud[df_completitud['periodo_contable'] == PERIODOS[-1]].sort_values('porcentaje').head(10)

286 filas (campo x periodo). Peores campos del último periodo:


,periodo_contable,total_registros,nombre_campo,cantidad_mala,tipo_indicador,porcentaje,es_total
12,202607,3230031,source_system,0,completitud,100.0,0
25,202607,3230031,accountable_period,0,completitud,100.0,0
38,202607,3230031,coverage_code,0,completitud,100.0,0
51,202607,3230031,branch_sk,0,completitud,100.0,0
64,202607,3230031,product_code,8,completitud,100.0,0
77,202607,3230031,transaction_date_sk,0,completitud,100.0,0
90,202607,3230031,policy_effective_date_sk,0,completitud,100.0,0
103,202607,3230031,policy_expiration_date_sk,0,completitud,100.0,0
116,202607,3230031,inception_date_sk,0,completitud,100.0,0
129,202607,3230031,transaction_type,0,completitud,100.0,0


In [13]:
# ---- EXACTITUD: valores dentro del dominio permitido (3 reglas implementadas).
# Nota deliberada: aquí SÍ se aplica el filtro compartido completo (incluye
# current_record_flag = 1), a diferencia del sql/03 original — hallazgo #1 del
# doc de hallazgos. Por eso las cifras pueden diferir levemente del original.
# Las reglas contra tabla ODS (transaction_delta_*) siguen sin implementar (hallazgo #2).
sql_exactitud = f'''
select
    accountable_period as periodo_contable,
    count(*) as total_registros,
    SUM(CASE WHEN source_system IS NULL OR source_system NOT IN ('CO_iaxis', 'CO_as400') THEN 1 ELSE 0 END) AS source_system,
    SUM(CASE WHEN current_record_flag IS NULL OR current_record_flag NOT IN (0, 1) THEN 1 ELSE 0 END) AS current_record_flag,
    SUM(CASE WHEN receipt_type IS NULL OR receipt_type NOT IN ('not-unificado', 'unificado-detail', 'unificado-total') THEN 1 ELSE 0 END) AS receipt_type
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where {FILTRO_BASE}
group by 1
order by 1;
'''

df_exactitud = a_formato_largo(run_query(sql_exactitud), 'exactitud')
df_exactitud

,periodo_contable,total_registros,nombre_campo,cantidad_mala,tipo_indicador,porcentaje,es_total
0,202507,4294122,source_system,0,exactitud,100.00,0
1,202508,3975901,source_system,0,exactitud,100.00,0
2,202509,4703622,source_system,0,exactitud,100.00,0
3,202510,4926896,source_system,0,exactitud,100.00,0
4,202511,4790756,source_system,0,exactitud,100.00,0
5,202512,4450874,source_system,0,exactitud,100.00,0
6,202601,4930056,source_system,0,exactitud,100.00,0
7,202602,4547533,source_system,0,exactitud,100.00,0
8,202603,4549776,source_system,0,exactitud,100.00,0
9,202604,4639530,source_system,0,exactitud,100.00,0


In [14]:
# ---- UNICIDAD: duplicados de policy_transaction_movement_sk dentro del universo filtrado.
sql_unicidad = f'''
with base as (
    select accountable_period as periodo_contable, policy_transaction_movement_sk
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where {FILTRO_BASE}
),
duplicados as (
    select periodo_contable, policy_transaction_movement_sk
    from base
    group by 1, 2
    having count(*) > 1
)
select
    b.periodo_contable,
    count(*) as total_registros,
    SUM(CASE WHEN d.policy_transaction_movement_sk IS NOT NULL THEN 1 ELSE 0 END) as cantidad_mala
from base b
left join duplicados d
  on d.periodo_contable = b.periodo_contable
 and d.policy_transaction_movement_sk = b.policy_transaction_movement_sk
group by 1
order by 1;
'''

df_unicidad = run_query(sql_unicidad)
df_unicidad['periodo_contable'] = df_unicidad['periodo_contable'].astype('int64')
df_unicidad['total_registros'] = df_unicidad['total_registros'].astype('int64')
df_unicidad['cantidad_mala'] = df_unicidad['cantidad_mala'].astype('int64')
df_unicidad['tipo_indicador'] = 'unicidad'
df_unicidad['nombre_campo'] = 'policy_transaction_movement_sk'
df_unicidad['porcentaje'] = (
    100.0 * (df_unicidad['total_registros'] - df_unicidad['cantidad_mala']) / df_unicidad['total_registros']
).round(2)
df_unicidad['es_total'] = 1
df_unicidad

,periodo_contable,total_registros,cantidad_mala,tipo_indicador,nombre_campo,porcentaje,es_total
0,202507,4294122,0,unicidad,policy_transaction_movement_sk,100.0,1
1,202508,3975901,0,unicidad,policy_transaction_movement_sk,100.0,1
2,202509,4703622,0,unicidad,policy_transaction_movement_sk,100.0,1
3,202510,4926896,0,unicidad,policy_transaction_movement_sk,100.0,1
4,202511,4790756,0,unicidad,policy_transaction_movement_sk,100.0,1
5,202512,4450874,0,unicidad,policy_transaction_movement_sk,100.0,1
6,202601,4930056,0,unicidad,policy_transaction_movement_sk,100.0,1
7,202602,4547533,0,unicidad,policy_transaction_movement_sk,100.0,1
8,202603,4549776,0,unicidad,policy_transaction_movement_sk,100.0,1
9,202604,4639530,0,unicidad,policy_transaction_movement_sk,100.0,1


In [15]:
# ---- VALIDEZ: formato/dominio por campo (12 reglas).
# movement_type está excluido a propósito (comentado también en el sql/05 original).
REGLAS_VALIDEZ = {
    'source_system': "source_system IS NULL OR source_system = 'Unknown'",
    'coverage_code': "coverage_code IS NULL OR coverage_code = 'Unknown'",
    'product_code': (
        "product_code IS NULL OR product_code = 'Unknown' OR LENGTH(product_code) > 6 "
        "OR product_code LIKE '% %' OR product_code ~ '[^A-Za-z0-9]' "
        "OR (source_system = 'CO_iaxis' AND product_code ~ '[A-Za-z]')"
    ),
    'transaction_date_sk': "transaction_date_sk IS NULL OR transaction_date_sk::VARCHAR !~ '^[0-9]{8}$'",
    'transaction_type': "transaction_type IS NULL OR transaction_type = 'Unknown' OR LENGTH(transaction_type) > 2",
    'current_record_flag': "current_record_flag IS NULL OR current_record_flag NOT IN (0, 1)",
    'risk_number': "risk_number IS NULL OR risk_number = 'Unknown'",
    'policy_number': "policy_number IS NULL OR policy_number = 'Unknown'",
    'accountable_period': "accountable_period IS NULL OR LENGTH(accountable_period::VARCHAR) <> 6",
    'policy_effective_date_sk': "policy_effective_date_sk IS NULL OR LENGTH(policy_effective_date_sk::VARCHAR) <> 8",
    'inception_date_sk': "inception_date_sk IS NULL OR LENGTH(inception_date_sk::VARCHAR) <> 8",
    'transaction_effective_date_sk': "transaction_effective_date_sk IS NULL OR LENGTH(transaction_effective_date_sk::VARCHAR) <> 8",
}

casos_validez = ',\n    '.join(
    f'SUM(CASE WHEN {regla} THEN 1 ELSE 0 END) AS {campo}'
    for campo, regla in REGLAS_VALIDEZ.items()
)

sql_validez = f'''
select
    accountable_period as periodo_contable,
    count(*) as total_registros,
    {casos_validez}
from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
where {FILTRO_BASE}
group by 1
order by 1;
'''

df_validez = a_formato_largo(run_query(sql_validez), 'validez')
print(f'{len(df_validez)} filas (campo x periodo). Peores campos del último periodo:')
df_validez[df_validez['periodo_contable'] == PERIODOS[-1]].sort_values('porcentaje').head(10)

156 filas (campo x periodo). Peores campos del último periodo:


,periodo_contable,total_registros,nombre_campo,cantidad_mala,tipo_indicador,porcentaje,es_total
12,202607,3230031,source_system,0,validez,100.0,0
25,202607,3230031,coverage_code,128,validez,100.0,0
38,202607,3230031,product_code,8,validez,100.0,0
51,202607,3230031,transaction_date_sk,0,validez,100.0,0
64,202607,3230031,transaction_type,0,validez,100.0,0
77,202607,3230031,current_record_flag,0,validez,100.0,0
90,202607,3230031,risk_number,0,validez,100.0,0
103,202607,3230031,policy_number,0,validez,100.0,0
116,202607,3230031,accountable_period,0,validez,100.0,0
129,202607,3230031,policy_effective_date_sk,0,validez,100.0,0


In [16]:
# ---- Verificación de acceso a la capa base (gde_adp_dwh) ----
# disponibilidad_sync compara tabla base vs. vista y requiere SELECT sobre
# gde_adp_dwh.fact_policy_transaction_movement. Esta sonda barata (limit 1) detecta
# si hay permiso ANTES de lanzar la query pesada; si no lo hay, ese chequeo se omite.
TIENE_ACCESO_DWH = False
try:
    run_query('select 1 from gde_adp_dwh.fact_policy_transaction_movement limit 1')
    TIENE_ACCESO_DWH = True
    print('OK: hay acceso a gde_adp_dwh.fact_policy_transaction_movement — se calculará disponibilidad_sync.')
except Exception as e:
    conn.rollback()  # la transacción queda abortada tras el error; sin esto fallan las celdas siguientes
    print(f'SIN ACCESO a gde_adp_dwh.fact_policy_transaction_movement ({type(e).__name__}).')
    print('disponibilidad_sync se omitirá; el resto de indicadores no se afecta.')

SIN ACCESO a gde_adp_dwh.fact_policy_transaction_movement (ProgrammingError).
disponibilidad_sync se omitirá; el resto de indicadores no se afecta.


In [17]:
# ---- DISPONIBILIDAD: dos chequeos NO comparables entre sí (ver CLAUDE.md y sql/06).
# A propósito SIN el filtro base compartido: compara tabla base vs. vista completas.

# (a) disponibilidad_sync: conciliación tabla base vs. vista (desfase de refresco).
#     ~98% es esperado/normal, no una falla.
#     OJO: requiere SELECT sobre la tabla base gde_adp_dwh.fact_policy_transaction_movement.
#     Si el usuario no tiene ese permiso (error 42501), el chequeo se omite con una
#     advertencia y el tablero lo muestra como "sin datos" — el resto sigue funcionando.
sql_disponibilidad_sync = f'''
with base_dwh as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           policy_transaction_movement_sk
    from gde_adp_dwh.fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in ({periodos_sql})
),
base_vista as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           policy_transaction_movement_sk
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in ({periodos_sql})
),
totales as (
    select periodo_contable, count(*) as total_registros
    from base_dwh
    group by 1
),
faltantes as (
    select d.periodo_contable, count(*) as cantidad_mala
    from base_dwh d
    left join base_vista v
      on v.periodo_contable = d.periodo_contable
     and v.policy_transaction_movement_sk = d.policy_transaction_movement_sk
    where v.policy_transaction_movement_sk is null
    group by 1
)
select t.periodo_contable, t.total_registros, coalesce(f.cantidad_mala, 0) as cantidad_mala
from totales t
left join faltantes f on f.periodo_contable = t.periodo_contable
order by 1;
'''

# (b) disponibilidad_regla4: % de ramos (product_code) con datos TODOS los días 1-15 del mes.
#     Solo usa la vista, así que no tiene el problema de permisos.
#     Porcentajes bajos son comunes (basta 1 día faltante para castigar el ramo).
#     Al mes EN CURSO solo se le exigen los días ya transcurridos (ver celda de
#     periodos); a los meses cerrados se les exigen los 15 completos.
_PEC = globals().get('PERIODO_EN_CURSO')
dias_exigidos_sql = (
    f'CASE WHEN periodo_contable = {_PEC} THEN {DIAS_REGLA4_EN_CURSO} ELSE 15 END'
    if _PEC else '15'
)
sql_disponibilidad_regla4 = f'''
with base as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           product_code,
           cast(substring(cast(transaction_date_sk as varchar), 7, 2) as integer) as dia
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in ({periodos_sql})
),
ramos as (
    select periodo_contable, product_code,
           count(distinct case when dia between 1 and 15 then dia end) as dias_presentes
    from base
    group by 1, 2
)
select
    periodo_contable,
    count(*) as total_registros,                                        -- total de ramos
    SUM(CASE WHEN dias_presentes >= {dias_exigidos_sql} THEN 0 ELSE 1 END) as cantidad_mala  -- ramos con algún día exigido faltante
from ramos
group by 1
order by 1;
'''

COLUMNAS_DISP = ['periodo_contable', 'total_registros', 'cantidad_mala',
                 'tipo_indicador', 'nombre_campo', 'porcentaje', 'es_total']
TIENE_ACCESO_DWH = globals().get('TIENE_ACCESO_DWH', False)  # False si no corrió la celda de verificación

partes = []
for campo, sql in [('disponibilidad_sync', sql_disponibilidad_sync),
                   ('disponibilidad_regla4', sql_disponibilidad_regla4)]:
    if campo == 'disponibilidad_sync' and not TIENE_ACCESO_DWH:
        print('AVISO: disponibilidad_sync omitido — sin permiso SELECT sobre gde_adp_dwh (ver celda anterior).')
        continue
    try:
        df = run_query(sql)
    except Exception as e:
        conn.rollback()  # la transacción queda abortada tras el error; sin esto fallan las celdas siguientes
        print(f'AVISO: {campo} omitido — {type(e).__name__}: {str(e)[:200]}')
        if campo == 'disponibilidad_sync':
            print('  (Este chequeo requiere SELECT sobre gde_adp_dwh.fact_policy_transaction_movement.')
            print('   Sin ese permiso no se puede comparar tabla base vs. vista; pídelo si necesitas el sync.)')
        continue
    df['periodo_contable'] = df['periodo_contable'].astype('int64')
    df['total_registros'] = df['total_registros'].astype('int64')
    df['cantidad_mala'] = df['cantidad_mala'].astype('int64')
    df['tipo_indicador'] = 'disponibilidad'
    df['nombre_campo'] = campo
    df['porcentaje'] = (
        100.0 * (df['total_registros'] - df['cantidad_mala']) / df['total_registros']
    ).round(2)
    df['es_total'] = 1
    partes.append(df[COLUMNAS_DISP])

if partes:
    df_disponibilidad = pd.concat(partes, ignore_index=True)
else:
    df_disponibilidad = pd.DataFrame(columns=COLUMNAS_DISP)
df_disponibilidad

AVISO: disponibilidad_sync omitido — sin permiso SELECT sobre gde_adp_dwh (ver celda anterior).


,periodo_contable,total_registros,cantidad_mala,tipo_indicador,nombre_campo,porcentaje,es_total
0,202507,88,81,disponibilidad,disponibilidad_regla4,7.95,1
1,202508,84,79,disponibilidad,disponibilidad_regla4,5.95,1
2,202509,88,79,disponibilidad,disponibilidad_regla4,10.23,1
3,202510,85,80,disponibilidad,disponibilidad_regla4,5.88,1
4,202511,83,77,disponibilidad,disponibilidad_regla4,7.23,1
5,202512,89,84,disponibilidad,disponibilidad_regla4,5.62,1
6,202601,85,82,disponibilidad,disponibilidad_regla4,3.53,1
7,202602,84,77,disponibilidad,disponibilidad_regla4,8.33,1
8,202603,82,76,disponibilidad,disponibilidad_regla4,7.32,1
9,202604,85,80,disponibilidad,disponibilidad_regla4,5.88,1


## 3) Consolidar el cubo en formato largo

Se unen los 5 indicadores en un solo DataFrame con el mismo grano del cubo unificado (`periodo_contable` + `tipo_indicador` + `nombre_campo`) y se agregan las filas de total por indicador (`TOTAL_PERIODO` / `TOTAL`, promedio simple de los porcentajes por campo — igual que el cubo).

In [18]:
COLUMNAS_CUBO = ['periodo_contable', 'tipo_indicador', 'nombre_campo',
                 'total_registros', 'cantidad_mala', 'porcentaje', 'es_total']

df_cubo = pd.concat(
    [df_completitud, df_exactitud, df_unicidad, df_validez, df_disponibilidad],
    ignore_index=True,
)[COLUMNAS_CUBO]

# Filas de total por indicador (promedio simple de los porcentajes por campo, como el cubo)
totales = []
for tipo, nombre in [('completitud', 'TOTAL_PERIODO'), ('exactitud', 'TOTAL'), ('validez', 'TOTAL_PERIODO')]:
    t = (df_cubo[df_cubo['tipo_indicador'] == tipo]
         .groupby('periodo_contable')
         .agg(total_registros=('total_registros', 'max'),
              cantidad_mala=('cantidad_mala', 'sum'),
              porcentaje=('porcentaje', 'mean'))
         .reset_index())
    t['porcentaje'] = t['porcentaje'].round(2)
    t['tipo_indicador'] = tipo
    t['nombre_campo'] = nombre
    t['es_total'] = 1
    totales.append(t[COLUMNAS_CUBO])

df_cubo = pd.concat([df_cubo] + totales, ignore_index=True).sort_values(
    ['periodo_contable', 'tipo_indicador', 'es_total', 'nombre_campo']
).reset_index(drop=True)

# Vista rápida: porcentaje total por indicador y periodo
df_cubo[df_cubo['es_total'] == 1].pivot_table(
    index=['tipo_indicador', 'nombre_campo'],
    columns='periodo_contable',
    values='porcentaje',
)

,periodo_contable,202507,202508,202509,202510,202511,202512,202601,202602,202603,202604,202605,202606,202607
tipo_indicador,nombre_campo,,,,,,,,,,,,,
completitud,TOTAL_PERIODO,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
disponibilidad,disponibilidad_regla4,7.95,5.95,10.23,5.88,7.23,5.62,3.53,8.33,7.32,5.88,3.85,9.30,7.04
exactitud,TOTAL,97.44,97.26,97.66,97.83,97.76,97.54,97.83,97.69,97.72,97.73,97.82,97.74,99.27
unicidad,policy_transaction_movement_sk,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
validez,TOTAL_PERIODO,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


In [19]:
# Guardar el cubo en CSV, junto al tablero HTML.
CSV_PATH = Path('..') / 'reports' / 'cubo_unificado.csv'
df_cubo.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f'CSV guardado en {CSV_PATH.resolve()} ({len(df_cubo)} filas).')

CSV guardado en C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\cubo_unificado.csv (546 filas).


## 4) Generar el tablero HTML de calidad

Se arma un archivo HTML **autocontenido** (sin dependencias externas, funciona sin internet) a partir de `df_cubo`, se guarda en `reports/tablero_calidad_primas.html` y se abre en el navegador.

El diseño replica la estructura del reporte Power BI original (`reportes_anteriores/Dashboard KPIs Calidad v2.pdf`), mejorado:

- **Mismas 5 pestañas**: Resumen ejecutivo · Detalle por variable · Tendencia e historia · Análisis de errores · Información.
- **Resumen ejecutivo**: anillos por dimensión + DQ Score (promedio de las 4 dimensiones comparables), tendencia del DQ Score vs Target 95, radar de dimensiones vs target y semáforo por dimensión. Disponibilidad aparece como tarjetas informativas aparte (sus 2 chequeos no son comparables con las demás dimensiones ni entre sí — no entran al DQ Score).
- **Detalle por variable**: calidad campo a campo por dimensión con barras 0–100 (sin eje truncado, a diferencia del PBI) y tabla consolidada con impacto.
- **Tendencia e historia**: evolución de las 4 dimensiones vs Target 95 + variación periodo a periodo (▲ Mejora / ▼ Baja / ● Estable).
- **Análisis de errores**: distribución de errores por dimensión, total por campo y Pareto 80/20 (barras y acumulado en un **solo eje** 0–100%, sin doble eje).
- **Mejoras vs el PBI**: selector de periodo que recalcula todo al instante, modo oscuro, tooltips, deltas vs periodo anterior y umbrales visibles, y **drill-down**: clic (o doble clic) en cualquier anillo del resumen abre una ventana con los campos exactos donde falla esa dimensión, con su definición de negocio (`DICCIONARIO_CAMPOS`).

Los umbrales de estado (`BUENO`/`ALERTA`/`GRAVE`/`CRÍTICO`) y el `TARGET` se ajustan en la celda siguiente. Disponibilidad tiene umbrales propios (ver `CLAUDE.md`).

In [20]:
# Target global del reporte (línea roja punteada del PBI original: "Target 95").
TARGET = 95.0

# Umbrales de estado por porcentaje (ajustables).
# Regla: >= BUENO -> BUENO; >= ALERTA -> ALERTA; >= GRAVE -> GRAVE; por debajo -> CRÍTICO.
#
# Disponibilidad tiene umbrales propios (ver CLAUDE.md):
#  - disponibilidad_sync: ~98% es esperado/normal (desfase de refresco vista vs. tabla), no una falla.
#  - disponibilidad_regla4: suele dar porcentajes bajos (basta 1 día faltante 1-15 para castigar el ramo).
UMBRALES = {
    'default':               {'BUENO': 99.0, 'ALERTA': 97.0, 'GRAVE': 90.0},
    'disponibilidad_sync':   {'BUENO': 97.0, 'ALERTA': 90.0, 'GRAVE': 80.0},
    'disponibilidad_regla4': {'BUENO': 80.0, 'ALERTA': 60.0, 'GRAVE': 40.0},
}

In [ ]:
# Reglas de cálculo por dimensión: alimentan el modal del tablero (clic en un anillo
# o en una tarjeta de Disponibilidad). Se construyen desde las MISMAS variables que
# arman los SELECT (CAMPOS_OBLIGATORIOS, REGLAS_VALIDEZ), así el SQL que se muestra
# nunca se desalinea del que realmente se ejecuta. En todas las dimensiones la regla
# cuenta registros MALOS (cantidad_mala); % = 100 × (1 − cantidad_mala / total_registros).

FILTRO_BASE_DOC = '''-- Universo DQ compartido (Completitud / Exactitud / Unicidad / Validez):
accountable_period IN (<periodos del tablero>)
  AND current_record_flag = 1
  AND coverage_code <> 8888
  AND transaction_delta_billed_premium_amount <> 0
  AND (
        (source_system = 'CO_iaxis' AND receipt_type NOT IN ('unificado-total'))
        OR source_system = 'CO_as400'
      )'''

_UNIVERSO_DOC = ('Movimientos del periodo con current_record_flag = 1 (versión vigente), cobertura '
                 'distinta de 8888, prima facturada distinta de 0, y recibos de iAxis que no sean '
                 '«unificado-total» (para no duplicar prima); AS400 entra completo.')

# Explicación en lenguaje natural de cada regla de Validez (mismas llaves que REGLAS_VALIDEZ).
VALIDEZ_QUE_HACE = {
    'source_system': "No debe ser nulo ni traer el valor comodín 'Unknown'.",
    'coverage_code': "No debe ser nulo ni traer el valor comodín 'Unknown'.",
    'product_code': ("No nulo ni 'Unknown', máximo 6 caracteres, sin espacios ni caracteres especiales. "
                     "Las letras solo se aceptan en AS400: en iAxis el código de producto debe ser numérico."),
    'transaction_date_sk': 'Debe ser un entero con forma de fecha YYYYMMDD (exactamente 8 dígitos).',
    'transaction_type': "No nulo ni 'Unknown' y con máximo 2 caracteres (es un código corto).",
    'current_record_flag': 'Solo admite 0 (histórico) o 1 (vigente); cualquier otro valor o nulo es inválido.',
    'risk_number': "No debe ser nulo ni traer el valor comodín 'Unknown'.",
    'policy_number': "No debe ser nulo ni traer el valor comodín 'Unknown'.",
    'accountable_period': 'Debe tener exactamente 6 dígitos con forma YYYYMM (p. ej. 202606).',
    'policy_effective_date_sk': 'Debe tener exactamente 8 dígitos con forma YYYYMMDD.',
    'inception_date_sk': 'Debe tener exactamente 8 dígitos con forma YYYYMMDD.',
    'transaction_effective_date_sk': 'Debe tener exactamente 8 dígitos con forma YYYYMMDD.',
}
assert set(VALIDEZ_QUE_HACE) == set(REGLAS_VALIDEZ), 'VALIDEZ_QUE_HACE desalineado con REGLAS_VALIDEZ'

REGLAS_DQ = {
    'completitud': {
        'explicacion': ('Por cada campo obligatorio se cuentan los registros del universo filtrado donde el '
                        'campo viene nulo (cantidad_mala = «Nulos»). El % del campo es 100 × (1 − nulos / '
                        'total_registros) y el anillo muestra el promedio simple de todos los campos '
                        '(fila TOTAL_PERIODO). Tres campos tienen excepciones por fuente porque AS400 no los '
                        'puebla: sseguro, receipt_type y receipt_number.'),
        'universo': _UNIVERSO_DOC,
        'universo_sql': FILTRO_BASE_DOC,
        'reglas': {
            **{
                c: {
                    'que_hace': 'El campo no debe venir nulo en ningún movimiento del universo filtrado.',
                    'sql': f'SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}',
                }
                for c in CAMPOS_OBLIGATORIOS
            },
            'sseguro': {
                'que_hace': ('Solo se exige en iAxis: AS400 no maneja el identificador interno sseguro, '
                             'así que sus nulos no se castigan.'),
                'sql': ("SUM(CASE WHEN sseguro IS NULL AND source_system <> 'CO_as400' "
                        'THEN 1 ELSE 0 END) AS sseguro'),
            },
            'receipt_type': {
                'que_hace': 'Solo se exige en iAxis: AS400 no puebla el tipo de recibo.',
                'sql': ("SUM(CASE WHEN receipt_type IS NULL AND source_system <> 'CO_as400' "
                        'THEN 1 ELSE 0 END) AS receipt_type'),
            },
            'receipt_number': {
                'que_hace': ('Regla doble: (a) en iAxis, los recibos que no son «not-unificado» deben traer '
                             'número de recibo; (b) al revés, un recibo «not-unificado» NO debe traer número '
                             '(si lo trae, también cuenta como error).'),
                'sql': """SUM(CASE
        WHEN (receipt_number IS NULL AND source_system <> 'CO_as400' AND receipt_type <> 'not-unificado')
          OR (receipt_type = 'not-unificado' AND receipt_number IS NOT NULL)
        THEN 1 ELSE 0
    END) AS receipt_number""",
            },
        },
    },
    'exactitud': {
        'explicacion': ('Cada campo debe traer valores dentro de su dominio permitido (lista cerrada de '
                        'valores válidos), no solo venir informado. Hay 3 reglas implementadas; las reglas '
                        'del original que comparan montos contra la tabla ODS siguen sin implementar '
                        '(hallazgo #2 del doc de hallazgos). A diferencia del sql/03 original, aquí SÍ se '
                        'aplica el filtro compartido completo con current_record_flag = 1 — desviación '
                        'deliberada documentada como hallazgo #1.'),
        'universo': _UNIVERSO_DOC,
        'universo_sql': FILTRO_BASE_DOC,
        'reglas': {
            'source_system': {
                'que_hace': "El sistema origen solo puede ser 'CO_iaxis' o 'CO_as400'; nulo u otro valor es inexacto.",
                'sql': ("SUM(CASE WHEN source_system IS NULL OR source_system NOT IN ('CO_iaxis', 'CO_as400') "
                        'THEN 1 ELSE 0 END) AS source_system'),
            },
            'current_record_flag': {
                'que_hace': 'La bandera de vigencia solo puede ser 0 o 1; nulo u otro valor es inexacto.',
                'sql': ('SUM(CASE WHEN current_record_flag IS NULL OR current_record_flag NOT IN (0, 1) '
                        'THEN 1 ELSE 0 END) AS current_record_flag'),
            },
            'receipt_type': {
                'que_hace': ("El tipo de recibo solo puede ser 'not-unificado', 'unificado-detail' o "
                             "'unificado-total'; nulo u otro valor es inexacto."),
                'sql': ("SUM(CASE WHEN receipt_type IS NULL OR receipt_type NOT IN "
                        "('not-unificado', 'unificado-detail', 'unificado-total') "
                        'THEN 1 ELSE 0 END) AS receipt_type'),
            },
        },
    },
    'unicidad': {
        'explicacion': ('La llave del movimiento (policy_transaction_movement_sk) no debe repetirse dentro '
                        'del periodo en el universo filtrado. cantidad_mala («Duplicados») cuenta TODOS los '
                        'registros que participan en un duplicado (si una llave aparece 2 veces, suma 2). '
                        '% = 100 × (1 − duplicados / total_registros); es una sola regla por periodo.'),
        'universo': _UNIVERSO_DOC,
        'universo_sql': FILTRO_BASE_DOC,
        'reglas': {
            'policy_transaction_movement_sk': {
                'que_hace': ('Se agrupa por periodo + llave; las llaves con más de una fila son duplicados '
                             'y todos sus registros cuentan como malos.'),
                'sql': """with base as (
    select accountable_period as periodo_contable, policy_transaction_movement_sk
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where <FILTRO_BASE>
),
duplicados as (
    select periodo_contable, policy_transaction_movement_sk
    from base
    group by 1, 2
    having count(*) > 1
)
select
    b.periodo_contable,
    count(*) as total_registros,
    SUM(CASE WHEN d.policy_transaction_movement_sk IS NOT NULL THEN 1 ELSE 0 END) as cantidad_mala
from base b
left join duplicados d
  on d.periodo_contable = b.periodo_contable
 and d.policy_transaction_movement_sk = b.policy_transaction_movement_sk
group by 1""",
            },
        },
    },
    'validez': {
        'explicacion': ('Cada campo debe cumplir su formato o dominio (longitudes, formas YYYYMMDD/YYYYMM, '
                        'caracteres permitidos, valores comodín prohibidos). La condición del CASE describe '
                        'al registro INVÁLIDO: si se cumple, suma 1 a cantidad_mala («Inválidos»). '
                        'movement_type está excluido a propósito, igual que en el sql/05 original.'),
        'universo': _UNIVERSO_DOC,
        'universo_sql': FILTRO_BASE_DOC,
        'reglas': {
            campo: {
                'que_hace': VALIDEZ_QUE_HACE[campo],
                'sql': f'SUM(CASE WHEN {regla} THEN 1 ELSE 0 END) AS {campo}',
            }
            for campo, regla in REGLAS_VALIDEZ.items()
        },
    },
    'dq': {
        'explicacion': ('Promedio simple de las 4 dimensiones comparables (Completitud, Exactitud, Unicidad '
                        'y Validez), tomando la fila TOTAL de cada una en el periodo seleccionado. '
                        'Disponibilidad no entra al DQ Score porque sus dos chequeos no son comparables '
                        'con el resto (miden sincronización y llegada de datos, no calidad campo a campo).'),
    },
    # Tarjetas informativas de Disponibilidad (clic sobre la tarjeta):
    'disponibilidad_sync': {
        'que_hace': ('Compara la tabla base del DWH contra la vista (SIN el filtro base compartido, a '
                     'propósito: cualquier filtro escondería huecos de sincronización). cantidad_mala son '
                     'los registros de la tabla que aún no aparecen en la vista — el desfase de refresco '
                     'de la vista materializada. Por eso ~98 % es esperado y normal, no una falla. '
                     'Requiere permiso SELECT sobre gde_adp_dwh; sin él, el chequeo se omite.'),
        'sql': """with base_dwh as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           policy_transaction_movement_sk
    from gde_adp_dwh.fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in (<periodos>)
),
base_vista as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           policy_transaction_movement_sk
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in (<periodos>)
),
totales as (
    select periodo_contable, count(*) as total_registros
    from base_dwh
    group by 1
),
faltantes as (
    select d.periodo_contable, count(*) as cantidad_mala
    from base_dwh d
    left join base_vista v
      on v.periodo_contable = d.periodo_contable
     and v.policy_transaction_movement_sk = d.policy_transaction_movement_sk
    where v.policy_transaction_movement_sk is null
    group by 1
)
select t.periodo_contable, t.total_registros, coalesce(f.cantidad_mala, 0) as cantidad_mala
from totales t
left join faltantes f on f.periodo_contable = t.periodo_contable""",
    },
    'disponibilidad_regla4': {
        'que_hace': ('Por cada ramo (product_code) verifica que hayan llegado datos TODOS los días 1 a 15 '
                     'del mes (día = posición 7-8 de transaction_date_sk). Un ramo con un solo día faltante '
                     'ya cuenta como malo, por eso los porcentajes bajos son comunes y no significan que '
                     'falte la mayoría de los datos. Al mes en curso solo se le exigen los días ya '
                     'transcurridos; a los meses cerrados, los 15 completos. También va SIN el filtro base.'),
        'sql': """with base as (
    select cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) as periodo_contable,
           product_code,
           cast(substring(cast(transaction_date_sk as varchar), 7, 2) as integer) as dia
    from gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement
    where cast(to_char(transaction_accounting_ts, 'YYYYMM') as integer) in (<periodos>)
),
ramos as (
    select periodo_contable, product_code,
           count(distinct case when dia between 1 and 15 then dia end) as dias_presentes
    from base
    group by 1, 2
)
select
    periodo_contable,
    count(*) as total_registros,   -- total de ramos
    SUM(CASE WHEN dias_presentes >= <dias exigidos: 15, o los transcurridos si el mes está en curso>
        THEN 0 ELSE 1 END) as cantidad_mala
from ramos
group by 1""",
    },
}
print(f"REGLAS_DQ listo: { {k: len(v.get('reglas', {})) for k, v in REGLAS_DQ.items()} }")

In [21]:
HTML_TEMPLATE = r'''<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>KPIs Calidad de Datos — CDP Primas</title>
<style>
  :root{
    /* Marca HDI (cromo del tablero, no codifica datos) */
    --brand:#0f7a3c; --brand-2:#8dc63f; --brand-soft:rgba(15,122,60,.10);
    /* Superficies y tinta */
    --surface:#fcfcfb; --page:#f4f4f1;
    --ink:#0b0b0b; --ink-2:#52514e; --muted:#898781;
    --grid:#e1e0d9; --border:rgba(11,11,11,.10); --track:#ecebe5;
    /* Estados (semáforo) */
    --good:#0ca30c; --warn:#fab219; --serious:#ec835a; --critical:#d03b3b;
    --delta-up:#006300; --delta-down:#d03b3b;
    /* Series categóricas (paleta validada de referencia, orden fijo) */
    --s1:#2a78d6; --s2:#eda100; --s3:#4a3aa7; --s4:#eb6834;
    --target:#d03b3b;
  }
  @media (prefers-color-scheme: dark){
    :root{
      --brand:#2fae62; --brand-2:#8dc63f; --brand-soft:rgba(47,174,98,.14);
      --surface:#1a1a19; --page:#0d0d0d;
      --ink:#ffffff; --ink-2:#c3c2b7;
      --grid:#2c2c2a; --border:rgba(255,255,255,.10); --track:#2c2c2a;
      --delta-up:#0ca30c;
      --s1:#3987e5; --s2:#c98500; --s3:#9085e9; --s4:#d95926;
    }
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--page);color:var(--ink);
       font:14px/1.45 system-ui,-apple-system,"Segoe UI",sans-serif}
  .wrap{max-width:1240px;margin:0 auto;padding:20px 24px 56px}

  /* ---- Encabezado estilo reporte PBI ---- */
  .hdr{display:flex;align-items:center;gap:18px;flex-wrap:wrap;
       background:var(--surface);border:1px solid var(--border);border-radius:12px;
       padding:14px 20px;margin-bottom:10px}
  .brand{display:flex;align-items:center;gap:10px}
  .bt b{display:block;font-size:19px;letter-spacing:.02em;color:var(--brand);line-height:1}
  .bt span{font-size:9.5px;letter-spacing:.28em;color:var(--ink-2)}
  .ttl{flex:1;min-width:260px;text-align:center}
  .ttl h1{font-size:21px;font-weight:700;color:var(--brand);margin:0}
  .ttl .sub{color:var(--muted);font-size:12px;margin:3px 0 0}
  .hctl{display:flex;flex-direction:column;gap:3px;align-items:flex-end}
  .hctl label{font-size:11.5px;color:var(--ink-2)}
  select{font:inherit;color:var(--ink);background:var(--surface);
         border:1px solid var(--border);border-radius:8px;padding:6px 12px;min-width:130px}
  .btn-refresh{font:inherit;font-size:12.5px;font-weight:600;cursor:pointer;
         color:#fff;background:var(--brand);border:none;border-radius:8px;
         padding:7px 14px;display:flex;align-items:center;gap:6px}
  .btn-refresh:hover{filter:brightness(1.08)}
  .btn-refresh svg{width:14px;height:14px}

  /* ---- Pestañas ---- */
  .tabs{display:flex;gap:8px;flex-wrap:wrap;margin:0 0 16px}
  .tab{font:inherit;font-size:13px;cursor:pointer;padding:8px 16px;border-radius:8px;
       border:1px solid var(--border);background:var(--surface);color:var(--ink-2)}
  .tab:hover{border-color:var(--brand)}
  .tab.on{background:var(--brand);border-color:var(--brand);color:#fff;font-weight:600}
  .pagina{display:none}
  .pagina.on{display:block}

  /* ---- Tarjetas / secciones ---- */
  section{background:var(--surface);border:1px solid var(--border);border-radius:12px;
          padding:18px 20px;margin-bottom:14px}
  section h2{font-size:15px;font-weight:650;margin:0 0 2px}
  section .nota{font-size:12.5px;color:var(--ink-2);margin:0 0 12px}
  .grid2{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:14px;align-items:start}
  .grid2 section{margin-bottom:0}

  /* ---- Anillos (resumen) ---- */
  .gauges{display:flex;gap:14px;flex-wrap:wrap;justify-content:space-between;margin-bottom:14px}
  .gauge{flex:1;min-width:168px;background:var(--surface);border:1px solid var(--border);
         border-radius:12px;padding:14px 10px;text-align:center}
  .gauge.dq{border-color:var(--brand);background:var(--brand-soft)}
  .gauge .glbl{font-size:12.5px;color:var(--ink-2);margin-bottom:8px;min-height:32px}
  .gauge .gdelta{font-size:12px;color:var(--muted);margin:6px 0}
  .gauge.clk{cursor:pointer;transition:border-color .15s, box-shadow .15s}
  .gauge.clk:hover{border-color:var(--brand);box-shadow:0 2px 12px rgba(0,0,0,.14)}
  .ghint{font-size:10.5px;color:var(--muted);margin-top:7px}
  .up{color:var(--delta-up);font-weight:600}
  .down{color:var(--delta-down);font-weight:600}
  .dispo{display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:14px;margin-bottom:14px}
  .dispo .tile{background:var(--surface);border:1px dashed var(--border);border-radius:12px;padding:14px 16px}
  .tile .lbl{font-size:12.5px;color:var(--ink-2)}
  .tile .val{font-size:26px;font-weight:600;margin:2px 0 4px}
  .tile .val small{font-size:14px;font-weight:500;color:var(--ink-2)}
  .tile .desc{font-size:11.5px;color:var(--muted);margin-top:6px}

  /* ---- Chips de estado ---- */
  .chip{display:inline-flex;align-items:center;gap:5px;font-size:11.5px;font-weight:600;
        padding:2px 9px;border-radius:999px;white-space:nowrap;
        background:color-mix(in srgb, var(--c) 14%, transparent)}
  .chip .ico{color:var(--c);font-size:11px}

  /* ---- Tablas ---- */
  .twrap{overflow-x:auto}
  table{border-collapse:collapse;width:100%;font-size:13px}
  th{color:var(--muted);font-weight:500;text-align:left;font-size:12px;
     padding:6px 10px;border-bottom:1px solid var(--grid);white-space:nowrap}
  th.num{text-align:right}
  td{padding:7px 10px;border-bottom:1px solid var(--grid)}
  tr:last-child td{border-bottom:none}
  td.num{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}
  td.campo{font-family:ui-monospace,Consolas,monospace;font-size:12.5px}
  table.verde thead th{background:var(--brand);color:#fff;font-weight:600;border-bottom:none}
  table.verde thead th:first-child{border-radius:8px 0 0 8px}
  table.verde thead th:last-child{border-radius:0 8px 8px 0}
  .track{height:8px;background:var(--track);border-radius:4px;min-width:100px}
  .fill{height:100%;background:var(--brand);border-radius:4px}
  .dot{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:7px;vertical-align:baseline}

  /* ---- Gráficos SVG ---- */
  svg.chart{width:100%;height:auto;display:block}
  svg text.ax{fill:var(--muted);font-size:10.5px;font-family:inherit}
  svg text.lb{fill:var(--ink-2);font-size:11px;font-family:inherit}
  svg text.big{fill:var(--ink);font-weight:600;font-family:inherit}
  .leyenda{display:flex;gap:14px;flex-wrap:wrap;font-size:12px;color:var(--ink-2);margin:2px 0 10px}
  .lg{display:inline-flex;align-items:center;gap:6px}
  .dash{display:inline-block;width:16px;border-top:2px dashed var(--target)}
  .hero{text-align:center;padding:28px 20px}
  .hero .hval{font-size:44px;font-weight:650;color:var(--brand);letter-spacing:-.01em}
  .hero .hlbl{font-size:13px;color:var(--ink-2);margin:4px 0 10px}

  td.campo[data-tip]{cursor:help;text-decoration:underline dotted;text-decoration-color:var(--ink-2);text-underline-offset:3px}

  /* ---- Tooltip ---- */
  #tip{position:fixed;display:none;z-index:10;pointer-events:none;max-width:280px;
       background:var(--ink);color:var(--surface);font-size:12px;line-height:1.4;
       padding:6px 10px;border-radius:8px;box-shadow:0 4px 14px rgba(0,0,0,.25)}

  /* ---- Modal drill-down (clic en un anillo del resumen) ---- */
  #modal{position:fixed;inset:0;display:none;z-index:30;background:rgba(0,0,0,.45);
         align-items:flex-start;justify-content:center;padding:6vh 16px}
  #modal.on{display:flex}
  .mbox{background:var(--surface);border:1px solid var(--border);border-radius:14px;
        max-width:900px;width:100%;max-height:84vh;overflow:auto;padding:0 22px 20px}
  .mhead{display:flex;justify-content:space-between;align-items:flex-start;gap:12px;
         position:sticky;top:0;background:var(--surface);padding:16px 0 8px;
         border-bottom:1px solid var(--grid);margin-bottom:10px}
  .mhead h2{font-size:16px;color:var(--brand);margin:0}
  .mclose{font:inherit;font-size:17px;line-height:1;cursor:pointer;background:none;
          border:none;color:var(--muted);padding:2px 6px}
  .mclose:hover{color:var(--ink)}
  tr.falla td{background:color-mix(in srgb, var(--critical) 9%, transparent)}
  td.dcampo{font-size:12px;color:var(--ink-2);min-width:220px;max-width:380px}
  .mbox h3{font-size:13.5px;color:var(--brand);margin:16px 0 6px}
  .mbox details{border:1px solid var(--border);border-radius:10px;padding:8px 12px;margin:8px 0}
  .mbox summary{cursor:pointer;font-size:13px}
  .mbox summary .campo{font-family:ui-monospace,Consolas,monospace;font-weight:600;font-size:12.5px}
  pre.sql{background:var(--page);border:1px solid var(--grid);border-radius:8px;
          padding:10px 12px;font:12px/1.5 ui-monospace,Consolas,monospace;
          overflow-x:auto;white-space:pre;color:var(--ink);margin:6px 0 2px}
  .dispo .tile.clk{cursor:pointer}
  .dispo .tile.clk:hover{border-color:var(--brand)}

  /* ---- Información ---- */
  .info-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:14px;align-items:start}
  .info-grid h3{font-size:13.5px;color:var(--brand);margin:14px 0 4px}
  .info-grid h3:first-child{margin-top:0}
  .info-grid p, .info-grid li{font-size:12.5px;color:var(--ink-2)}
  .info-grid ul{margin:4px 0;padding-left:18px}
  .kv{font-size:12.5px;color:var(--ink-2)}
  .kv b{color:var(--ink)}
  footer{color:var(--muted);font-size:11.5px;margin-top:18px;text-align:center}
</style>
</head>
<body>
<div id="tip"></div>
<div id="modal"><div class="mbox" id="mbox"></div></div>
<div class="wrap">

  <header class="hdr">
    <div class="brand">
      <svg width="36" height="36" viewBox="0 0 36 36" aria-hidden="true">
        <rect x="3"  y="12" width="12" height="12" fill="var(--brand-2)"/>
        <rect x="15" y="2"  width="9"  height="9"  fill="var(--brand)"/>
        <rect x="21" y="14" width="12" height="12" fill="var(--brand)"/>
        <rect x="12" y="25" width="8"  height="8"  fill="var(--brand-2)"/>
      </svg>
      <div class="bt"><b>HDI</b><span>SEGUROS</span></div>
    </div>
    <div class="ttl">
      <h1>KPIs Calidad de Datos — CDP Primas</h1>
      <p class="sub" id="sub"></p>
    </div>
    <div class="hctl">
      <label for="periodo">Periodo contable</label>
      <select id="periodo"></select>
    </div>
    <button class="btn-refresh" id="btnRefresh" type="button">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"><path d="M3 12a9 9 0 0 1 15.4-6.4M21 12a9 9 0 0 1-15.4 6.4"/><path d="M18 3v5h-5M6 21v-5h5"/></svg>
      Actualizar informe
    </button>
  </header>

  <nav class="tabs" id="tabs"></nav>

  <div class="pagina" id="pg-resumen">
    <div class="gauges" id="gauges"></div>
    <div class="dispo" id="dispo"></div>
    <div class="grid2">
      <section>
        <h2>Tendencia — Comportamiento general (DQ Score)</h2>
        <p class="nota">Promedio simple de las 4 dimensiones comparables vs. Target.</p>
        <div id="dq-trend"></div>
      </section>
      <section>
        <h2>Radar de dimensiones vs Target</h2>
        <p class="nota">Escala 75–100. La línea punteada roja es el target.</p>
        <div id="radar" style="display:flex;justify-content:center"></div>
      </section>
    </div>
    <section>
      <h2>Semáforo por dimensión</h2>
      <p class="nota">Detalle numérico de los anillos (vista de tabla del resumen).</p>
      <div id="semaforo"></div>
    </section>
  </div>

  <div class="pagina" id="pg-detalle"></div>

  <div class="pagina" id="pg-tendencia">
    <section>
      <h2>Evolución histórica de KPIs por dimensión</h2>
      <div class="leyenda" id="evo-leyenda"></div>
      <div id="evolucion"></div>
    </section>
    <section>
      <h2>Variación periodo a periodo</h2>
      <p class="nota">Periodo seleccionado vs. el inmediatamente anterior.</p>
      <div id="variacion"></div>
    </section>
  </div>

  <div class="pagina" id="pg-errores"></div>

  <div class="pagina" id="pg-info"></div>

  <footer id="pie"></footer>
</div>

<script>
const DATA = __DATA__;
const P = DATA.periodos;
const ENCURSO = DATA.periodo_en_curso || null;
const F = DATA.filas;
const TARGET = DATA.target;
const fmtInt = new Intl.NumberFormat('es-CO').format;
const DICC = DATA.diccionario || {};
const REGLAS = DATA.reglas || {};

/* Dimensiones comparables (entran al DQ Score). Colores: paleta categórica validada, orden fijo. */
const DIMS = [
  {tipo:'completitud', total:'TOTAL_PERIODO', label:'Completitud', malos:'Nulos', color:'var(--s1)',
   desc:'% de registros no nulos por campo obligatorio (promedio simple de los campos).'},
  {tipo:'exactitud', total:'TOTAL', label:'Exactitud', malos:'Inexactos', color:'var(--s2)',
   desc:'Valores dentro del dominio permitido por campo.'},
  {tipo:'unicidad', total:'policy_transaction_movement_sk', label:'Unicidad', malos:'Duplicados', color:'var(--s3)',
   desc:'Registros sin duplicado de policy_transaction_movement_sk.'},
  {tipo:'validez', total:'TOTAL_PERIODO', label:'Validez', malos:'Inválidos', color:'var(--s4)',
   desc:'Formato/dominio por campo (longitudes, formas YYYYMMDD/YYYYMM, caracteres permitidos).'},
];
/* Disponibilidad: informativa, fuera del DQ Score (chequeos no comparables). */
const DISPO = [
  {campo:'disponibilidad_sync', label:'Disponibilidad — sincronización', umbral:'disponibilidad_sync',
   desc:'Tabla base vs. vista (desfase de refresco de la vista materializada); ~98% es esperado y normal, no una falla.'},
  {campo:'disponibilidad_regla4', label:'Disponibilidad — regla 4 (días 1–15)', umbral:'disponibilidad_regla4',
   desc:'% de ramos (product_code) con datos todos los días 1–15 del mes; porcentajes bajos son comunes (1 día faltante castiga el ramo).'},
];
const ESTADOS = {
  'BUENO':   {c:'var(--good)',     i:'✓', imp:'OK'},
  'ALERTA':  {c:'var(--warn)',     i:'▲', imp:'Alerta'},
  'GRAVE':   {c:'var(--serious)',  i:'●', imp:'Grave'},
  'CRÍTICO': {c:'var(--critical)', i:'✕', imp:'Crítico'},
};

/* ---------- helpers de datos ---------- */
function fila(tipo, campo, per){
  return F.find(f => f.tipo_indicador === tipo && f.nombre_campo === campo && f.periodo_contable === per);
}
function filaTotalDim(d, per){ return fila(d.tipo, d.total, per); }
function detalleDim(tipo, per){
  return F.filter(f => f.tipo_indicador === tipo && f.periodo_contable === per && !f.es_total);
}
function serieDim(d){ return P.map(p => { const f = filaTotalDim(d, p); return f ? f.porcentaje : null; }); }
function dqScore(per){
  const vs = DIMS.map(d => filaTotalDim(d, per)).filter(Boolean).map(f => f.porcentaje);
  return vs.length ? vs.reduce((a,b) => a+b, 0) / vs.length : null;
}
function estado(clave, pct){
  const u = DATA.umbrales[clave] || DATA.umbrales['default'];
  if (pct >= u.BUENO) return 'BUENO';
  if (pct >= u.ALERTA) return 'ALERTA';
  if (pct >= u.GRAVE) return 'GRAVE';
  return 'CRÍTICO';
}
function chip(e, txt){
  const s = ESTADOS[e];
  return '<span class="chip" style="--c:'+s.c+'"><span class="ico">'+s.i+'</span>'+(txt || e)+'</span>';
}
function barra(p){
  const w = Math.max(0, Math.min(100, p));
  return '<div class="track" data-tip="'+p.toFixed(2)+' % (escala completa 0–100)"><div class="fill" style="width:'+w+'%"></div></div>';
}

/* Celda de nombre de campo: al pasar el mouse muestra la definición del
   diccionario (vw_fact_policy_transaction_movement.csv) vía el tooltip data-tip. */
function escAttr(s){ return String(s).replace(/&/g,'&amp;').replace(/"/g,'&quot;').replace(/</g,'&lt;'); }
function escHtml(s){ return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;'); }
function tdCampo(nombre){
  const d = DICC[nombre];
  return '<td class="campo"'+(d ? ' data-tip="'+escAttr(d)+'"' : '')+'>'+nombre+'</td>';
}

/* ---------- componentes SVG ---------- */
function gauge(pct, color, size){
  const st = 11, r = (size - st) / 2 - 1, c = 2 * Math.PI * r, half = size / 2;
  const off = c * (1 - Math.max(0, Math.min(100, pct == null ? 0 : pct)) / 100);
  let g = '<g transform="rotate(-90 '+half+' '+half+')">'
    + '<circle cx="'+half+'" cy="'+half+'" r="'+r+'" fill="none" stroke="var(--track)" stroke-width="'+st+'"/>';
  if (pct != null){
    g += '<circle cx="'+half+'" cy="'+half+'" r="'+r+'" fill="none" stroke="'+color+'" stroke-width="'+st+'"'
      + ' stroke-linecap="round" stroke-dasharray="'+c.toFixed(1)+'" stroke-dashoffset="'+off.toFixed(1)+'"/>';
  }
  g += '</g><text x="'+half+'" y="'+(half+6)+'" text-anchor="middle" class="big" font-size="'+(size>=130?19:16)+'">'
    + (pct == null ? '—' : pct.toFixed(2)+'%') + '</text>';
  return '<svg width="'+size+'" height="'+size+'" data-tip="'+(pct == null ? 'Sin datos' : pct.toFixed(2)+' %')+'">'+g+'</svg>';
}

function lineChart(series, opts){
  opts = opts || {};
  const w = opts.w || 860, h = opts.h || 280;
  const pad = {t:16, r:opts.padR || 150, b:30, l:42};
  const nums = [];
  series.forEach(s => s.vals.forEach(v => { if (v != null) nums.push(v); }));
  if (TARGET != null) nums.push(TARGET);
  if (!nums.length) return '<p class="nota">Sin datos.</p>';
  let yMin = Math.floor((Math.min.apply(null, nums) - 1) / 5) * 5;
  if (yMin < 0) yMin = 0;
  const yMax = 100, span = (yMax - yMin) || 1;
  const x = i => pad.l + (w - pad.l - pad.r) * (P.length > 1 ? i / (P.length - 1) : 0.5);
  const y = v => pad.t + (h - pad.t - pad.b) * (1 - (v - yMin) / span);
  const step = span <= 25 ? 5 : 10;
  let g = '';
  for (let v = yMin; v <= yMax; v += step){
    g += '<line x1="'+pad.l+'" x2="'+(w-pad.r)+'" y1="'+y(v)+'" y2="'+y(v)+'" stroke="var(--grid)"/>'
       + '<text x="'+(pad.l-8)+'" y="'+(y(v)+4)+'" text-anchor="end" class="ax">'+v+'</text>';
  }
  P.forEach((p, i) => { g += '<text x="'+x(i)+'" y="'+(h-8)+'" text-anchor="middle" class="ax">'+p+'</text>'; });
  if (TARGET != null){
    g += '<line x1="'+pad.l+'" x2="'+(w-pad.r)+'" y1="'+y(TARGET)+'" y2="'+y(TARGET)+'"'
       + ' stroke="var(--target)" stroke-width="1.5" stroke-dasharray="4 4"/>';
  }
  const endLabels = [];
  series.forEach(s => {
    const pts = s.vals.map((v, i) => ({v, i})).filter(d => d.v != null);
    if (!pts.length) return;
    if (pts.length > 1){
      const poly = pts.map(d => x(d.i).toFixed(1)+','+y(d.v).toFixed(1)).join(' ');
      g += '<polyline points="'+poly+'" fill="none" stroke="'+s.color+'" stroke-width="2"'
         + ' stroke-linecap="round" stroke-linejoin="round"/>';
    }
    pts.forEach(d => {
      g += '<circle cx="'+x(d.i)+'" cy="'+y(d.v)+'" r="3.5" fill="'+s.color+'" stroke="var(--surface)" stroke-width="1.5"/>'
         + '<circle cx="'+x(d.i)+'" cy="'+y(d.v)+'" r="12" fill="transparent"'
         + ' data-tip="'+s.label+' · '+P[d.i]+': '+d.v.toFixed(2)+' %"/>';
    });
    const last = pts[pts.length - 1];
    endLabels.push({y: y(last.v), text: s.label+' '+last.v.toFixed(2), color: s.color});
  });
  if (TARGET != null) endLabels.push({y: y(TARGET), text: 'Target '+TARGET, color: 'var(--target)'});
  endLabels.sort((a, b) => a.y - b.y);
  for (let i = 1; i < endLabels.length; i++){
    if (endLabels[i].y - endLabels[i-1].y < 14) endLabels[i].y = endLabels[i-1].y + 14;
  }
  endLabels.forEach(l => {
    g += '<text x="'+(w-pad.r+10)+'" y="'+(l.y+4)+'" class="lb"><tspan fill="'+l.color+'">●</tspan> '+l.text+'</text>';
  });
  return '<svg viewBox="0 0 '+w+' '+h+'" class="chart">'+g+'</svg>';
}

function radar(per){
  /* Ejes como el PBI: Unicidad arriba, Completitud derecha, Validez abajo, Exactitud izquierda */
  const orden = [DIMS[2], DIMS[0], DIMS[3], DIMS[1]];
  const size = 300, cx = size/2, cy = size/2, R = 100, MIN = 75;
  const ang = [-90, 0, 90, 180].map(a => a * Math.PI / 180);
  const rr = v => R * Math.max(0, Math.min(100, v) - MIN) / (100 - MIN);
  const pt = (a, v) => [cx + Math.cos(a) * rr(v), cy + Math.sin(a) * rr(v)];
  const poly = vs => ang.map((a, i) => pt(a, vs[i]).map(n => n.toFixed(1)).join(',')).join(' ');
  let g = '';
  for (let v = 80; v <= 100; v += 5){
    g += '<polygon points="'+poly([v,v,v,v])+'" fill="none" stroke="var(--grid)"/>';
    if (v % 10 === 0) g += '<text x="'+(cx+5)+'" y="'+(cy-rr(v)-3)+'" class="ax">'+v+'</text>';
  }
  ang.forEach(a => { const p = pt(a, 100); g += '<line x1="'+cx+'" y1="'+cy+'" x2="'+p[0]+'" y2="'+p[1]+'" stroke="var(--grid)"/>'; });
  if (TARGET != null){
    g += '<polygon points="'+poly([TARGET,TARGET,TARGET,TARGET])+'" fill="none"'
       + ' stroke="var(--target)" stroke-width="1.5" stroke-dasharray="4 4"/>';
  }
  const vals = orden.map(d => { const f = filaTotalDim(d, per); return f ? f.porcentaje : null; });
  const vSafe = vals.map(v => v == null ? MIN : v);
  g += '<polygon points="'+poly(vSafe)+'" fill="var(--brand)" fill-opacity=".16" stroke="var(--brand)" stroke-width="2"/>';
  ang.forEach((a, i) => {
    if (vals[i] == null) return;
    const p = pt(a, vals[i]);
    g += '<circle cx="'+p[0].toFixed(1)+'" cy="'+p[1].toFixed(1)+'" r="4.5" fill="var(--brand)"'
       + ' stroke="var(--surface)" stroke-width="2" data-tip="'+orden[i].label+': '+vals[i].toFixed(2)+' %"/>';
  });
  const lbl = [
    [cx, cy - R - 10, 'middle'], [cx + R + 10, cy + 4, 'start'],
    [cx, cy + R + 18, 'middle'], [cx - R - 10, cy + 4, 'end'],
  ];
  orden.forEach((d, i) => {
    g += '<text x="'+lbl[i][0]+'" y="'+lbl[i][1]+'" text-anchor="'+lbl[i][2]+'" class="lb">'+d.label+'</text>';
  });
  return '<svg width="'+size+'" height="'+(size+10)+'" style="max-width:100%">'+g+'</svg>';
}

function donutDist(items, total){
  const size = 190, st = 24, half = size/2, r = half - st/2 - 2, c = 2 * Math.PI * r, gap = 2;
  let acc = 0, g = '';
  items.filter(it => it.val > 0).forEach(it => {
    const len = c * it.val / total, seg = Math.max(0.5, len - gap);
    g += '<circle cx="'+half+'" cy="'+half+'" r="'+r+'" fill="none" stroke="'+it.color+'" stroke-width="'+st+'"'
       + ' stroke-dasharray="'+seg.toFixed(1)+' '+(c-seg).toFixed(1)+'" stroke-dashoffset="'+(-acc).toFixed(1)+'"'
       + ' data-tip="'+it.label+': '+fmtInt(it.val)+' errores ('+(100*it.val/total).toFixed(2)+' %)"/>';
    acc += len;
  });
  return '<svg width="'+size+'" height="'+size+'">'
    + '<g transform="rotate(-90 '+half+' '+half+')">'+g+'</g>'
    + '<text x="'+half+'" y="'+(half-2)+'" text-anchor="middle" class="big" font-size="17">'+fmtInt(total)+'</text>'
    + '<text x="'+half+'" y="'+(half+15)+'" text-anchor="middle" class="ax">errores</text></svg>';
}

function pareto(top, total){
  const w = 860, h = 320, pad = {t:30, r:20, b:64, l:46};
  const n = top.length;
  const bw = Math.min(60, (w - pad.l - pad.r) / n * 0.62);
  const x = i => pad.l + (w - pad.l - pad.r) * (i + 0.5) / n;
  const y = v => pad.t + (h - pad.t - pad.b) * (1 - v / 100);
  let g = '';
  [0, 25, 50, 75, 100].forEach(v => {
    g += '<line x1="'+pad.l+'" x2="'+(w-pad.r)+'" y1="'+y(v)+'" y2="'+y(v)+'" stroke="var(--grid)"/>'
       + '<text x="'+(pad.l-8)+'" y="'+(y(v)+4)+'" text-anchor="end" class="ax">'+v+'%</text>';
  });
  let cum = 0;
  const linePts = [];
  top.forEach((t, i) => {
    const share = 100 * t[1] / total;
    cum += share;
    linePts.push([x(i), y(cum), cum]);
    g += '<rect x="'+(x(i)-bw/2)+'" y="'+y(share)+'" width="'+bw+'" height="'+Math.max(1,(y(0)-y(share)))+'"'
       + ' rx="3" fill="var(--brand)" data-tip="'+t[0]+': '+fmtInt(t[1])+' errores ('+share.toFixed(2)+' % del total)"/>'
       + '<text x="'+x(i)+'" y="'+(y(share)-6)+'" text-anchor="middle" class="ax">'+fmtInt(t[1])+'</text>';
    const nom = t[0].length > 14 ? t[0].slice(0, 13)+'…' : t[0];
    g += '<text x="'+x(i)+'" y="'+(h-pad.b+16)+'" text-anchor="end" class="ax"'
       + ' transform="rotate(-30 '+x(i)+' '+(h-pad.b+16)+')">'+nom+'</text>';
  });
  if (linePts.length > 1){
    g += '<polyline points="'+linePts.map(p => p[0].toFixed(1)+','+p[1].toFixed(1)).join(' ')+'"'
       + ' fill="none" stroke="var(--s1)" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>';
  }
  linePts.forEach((p, i) => {
    g += '<circle cx="'+p[0]+'" cy="'+p[1]+'" r="3.5" fill="var(--s1)" stroke="var(--surface)" stroke-width="1.5"/>'
       + '<circle cx="'+p[0]+'" cy="'+p[1]+'" r="12" fill="transparent" data-tip="Acumulado hasta '+top[i][0]+': '+p[2].toFixed(2)+' %"/>';
  });
  const lastP = linePts[linePts.length - 1];
  g += '<text x="'+(lastP[0]+8)+'" y="'+(lastP[1]-8)+'" class="lb"><tspan fill="var(--s1)">●</tspan> acumulado '+lastP[2].toFixed(0)+'%</text>';
  return '<svg viewBox="0 0 '+w+' '+h+'" class="chart">'+g+'</svg>';
}

/* ---------- render por pestaña ---------- */
function renderResumen(per){
  const idx = P.indexOf(per);
  let gs = '';
  DIMS.forEach(d => {
    const f = filaTotalDim(d, per);
    const pct = f ? f.porcentaje : null;
    const e = pct == null ? null : estado('default', pct);
    let delta = '&nbsp;';
    if (pct != null && idx > 0){
      const fp = filaTotalDim(d, P[idx-1]);
      if (fp){
        const dd = pct - fp.porcentaje;
        delta = '<span class="'+(dd >= 0 ? 'up' : 'down')+'">'+(dd >= 0 ? '+' : '')+dd.toFixed(2)+' pp</span> vs '+P[idx-1];
      }
    }
    gs += '<div class="gauge clk" data-dim="'+d.tipo+'" title="Ver en qué campos falla '+d.label+'"><div class="glbl">'+d.label+'</div>'
        + gauge(pct, e ? ESTADOS[e].c : 'var(--muted)', 116)
        + '<div class="gdelta">'+delta+'</div>'
        + (e ? chip(e) : '<span class="chip" style="--c:var(--muted)">sin datos</span>')
        + '<div class="ghint">&#128269; clic: campos y reglas</div></div>';
  });
  const dq = dqScore(per);
  let dqDelta = '&nbsp;';
  if (dq != null && idx > 0){
    const dqAnt = dqScore(P[idx-1]);
    if (dqAnt != null){
      const dd = dq - dqAnt;
      dqDelta = '<span class="'+(dd >= 0 ? 'up' : 'down')+'">'+(dd >= 0 ? '+' : '')+dd.toFixed(2)+' pp</span> vs '+P[idx-1];
    }
  }
  gs += '<div class="gauge dq clk" data-dim="dq" title="Ver el resumen de fallas por dimensión"><div class="glbl"><strong>Comportamiento general</strong><br>(DQ Score)</div>'
      + gauge(dq, 'var(--brand)', 130)
      + '<div class="gdelta">'+dqDelta+'</div>'
      + (dq != null ? chip(estado('default', dq)) : '')
      + '<div class="ghint">&#128269; clic: fallas por dimensión</div></div>';
  document.getElementById('gauges').innerHTML = gs;

  document.getElementById('dispo').innerHTML = DISPO.map(dp => {
    const f = fila('disponibilidad', dp.campo, per);
    let cuerpo;
    if (f){
      const e = estado(dp.umbral, f.porcentaje);
      cuerpo = '<div class="val">'+f.porcentaje.toFixed(2)+'<small> %</small></div>'+chip(e);
    } else {
      cuerpo = '<div class="val">—</div><span class="chip" style="--c:var(--muted)">sin datos</span>';
      if (dp.campo === 'disponibilidad_sync'){
        dp = Object.assign({}, dp, {desc: dp.desc+' ⚠ Omitido en esta corrida: el usuario no tiene SELECT sobre gde_adp_dwh.fact_policy_transaction_movement.'});
      }
    }
    return '<div class="tile clk" data-dispo="'+dp.campo+'" title="Ver la regla de cálculo"><div class="lbl">'+dp.label+' <em>(informativa — no entra al DQ Score)</em></div>'
      + cuerpo + '<div class="desc">'+dp.desc+'</div>'
      + '<div class="ghint">&#128269; clic: regla de cálculo</div></div>';
  }).join('');

  const dqSerie = P.map(p => { const v = dqScore(p); return v == null ? null : Math.round(v * 100) / 100; });
  document.getElementById('dq-trend').innerHTML =
    lineChart([{label:'DQ Score', color:'var(--brand)', vals:dqSerie}], {h:260, padR:140});
  document.getElementById('radar').innerHTML = radar(per);

  const rows = DIMS.map(d => {
    const f = filaTotalDim(d, per);
    if (!f) return '';
    const campos = detalleDim(d.tipo, per).length || 1;
    return '<tr><td><span class="dot" style="background:'+d.color+'"></span>'+d.label+'</td>'
      + '<td class="num">'+f.porcentaje.toFixed(2)+'</td>'
      + '<td class="num">'+campos+'</td>'
      + '<td class="num">'+fmtInt(f.total_registros)+'</td>'
      + '<td>'+barra(f.porcentaje)+'</td>'
      + '<td>'+chip(estado('default', f.porcentaje))+'</td></tr>';
  }).join('');
  document.getElementById('semaforo').innerHTML =
    '<div class="twrap"><table class="verde"><thead><tr><th>Dimensión</th><th class="num">% Calidad</th>'
    + '<th class="num">Campos</th><th class="num">Total registros</th><th style="width:150px"></th><th>Estado</th>'
    + '</tr></thead><tbody>'+rows+'</tbody></table></div>';
}

function renderDetalle(per){
  let html = '<div class="grid2">';
  [DIMS[0], DIMS[3], DIMS[1]].forEach(d => {   // Completitud, Validez, Exactitud (como el PBI)
    const det = detalleDim(d.tipo, per).slice().sort((a, b) => a.porcentaje - b.porcentaje);
    const rows = det.map(f => '<tr>'+tdCampo(f.nombre_campo)
      + '<td>'+barra(f.porcentaje)+'</td>'
      + '<td class="num">'+f.porcentaje.toFixed(2)+'</td>'
      + '<td class="num">'+fmtInt(f.cantidad_mala)+'</td>'
      + '<td>'+chip(estado('default', f.porcentaje))+'</td></tr>').join('');
    html += '<section><h2>'+d.label+' por campo</h2><p class="nota">'+d.desc+'</p>'
      + (det.length
          ? '<div class="twrap"><table><thead><tr><th>Campo</th><th style="width:130px">Escala 0–100</th>'
            + '<th class="num">%</th><th class="num">'+d.malos+'</th><th>Estado</th></tr></thead><tbody>'+rows+'</tbody></table></div>'
          : '<p class="nota">Sin datos para este periodo.</p>')
      + '</section>';
  });
  const fu = filaTotalDim(DIMS[2], per);
  html += '<section class="hero"><h2>Unicidad</h2>'
    + '<div class="hval">'+(fu ? fu.porcentaje.toFixed(2)+'%' : '—')+'</div>'
    + '<div class="hlbl">policy_transaction_movement_sk sin duplicados'
    + (fu ? ' · '+fmtInt(fu.cantidad_mala)+' duplicados de '+fmtInt(fu.total_registros)+' registros' : '')+'</div>'
    + (fu ? chip(estado('default', fu.porcentaje)) : '')+'</section>';
  html += '</div>';

  const all = [];
  DIMS.forEach(d => {
    let det = detalleDim(d.tipo, per);
    if (!det.length){ const f = filaTotalDim(d, per); det = f ? [f] : []; }
    det.forEach(f => all.push({campo: f.nombre_campo, dim: d.label, pct: f.porcentaje, err: f.cantidad_mala}));
  });
  all.sort((a, b) => a.pct - b.pct || b.err - a.err);
  const rows = all.map(r => {
    const e = estado('default', r.pct);
    return '<tr>'+tdCampo(r.campo)+'<td>'+r.dim+'</td>'
      + '<td class="num">'+r.pct.toFixed(2)+'</td><td class="num">'+fmtInt(r.err)+'</td>'
      + '<td>'+chip(e, ESTADOS[e].imp)+'</td></tr>';
  }).join('');
  html += '<section style="margin-top:14px"><h2>Calidad por campo (todas las dimensiones)</h2>'
    + '<p class="nota">Ordenada de peor a mejor — lo primero de la lista es lo primero a remediar.</p>'
    + '<div class="twrap"><table class="verde"><thead><tr><th>Campo</th><th>Dimensión</th>'
    + '<th class="num">% Calidad</th><th class="num">Registros con error</th><th>Impacto</th>'
    + '</tr></thead><tbody>'+rows+'</tbody></table></div></section>';
  document.getElementById('pg-detalle').innerHTML = html;
}

function renderTendencia(per){
  const series = DIMS.map(d => ({label: d.label, color: d.color, vals: serieDim(d)}));
  document.getElementById('evolucion').innerHTML = lineChart(series, {h:310, padR:175});
  document.getElementById('evo-leyenda').innerHTML =
    series.map(s => '<span class="lg"><span class="dot" style="background:'+s.color+'"></span>'+s.label+'</span>').join('')
    + '<span class="lg"><span class="dash"></span>Target '+TARGET+'</span>';

  const idx = P.indexOf(per);
  const rows = DIMS.map(d => {
    const act = filaTotalDim(d, per);
    const ant = idx > 0 ? filaTotalDim(d, P[idx-1]) : null;
    if (!act) return '';
    let varTxt = '—', estTxt = '<span class="chip" style="--c:var(--muted)">sin anterior</span>';
    if (ant){
      const dd = act.porcentaje - ant.porcentaje;
      varTxt = (dd >= 0 ? '+' : '')+dd.toFixed(2);
      if (Math.abs(dd) < 0.005) estTxt = '<span class="chip" style="--c:var(--muted)"><span class="ico">●</span>Estable</span>';
      else if (dd > 0) estTxt = '<span class="chip" style="--c:var(--delta-up)"><span class="ico">▲</span>Mejora</span>';
      else estTxt = '<span class="chip" style="--c:var(--delta-down)"><span class="ico">▼</span>Baja</span>';
    }
    return '<tr><td><span class="dot" style="background:'+d.color+'"></span>'+d.label+'</td>'
      + '<td class="num">'+act.porcentaje.toFixed(2)+'</td>'
      + '<td class="num">'+(ant ? ant.porcentaje.toFixed(2) : '—')+'</td>'
      + '<td class="num">'+varTxt+'</td><td>'+estTxt+'</td></tr>';
  }).join('');
  document.getElementById('variacion').innerHTML =
    '<div class="twrap"><table class="verde"><thead><tr><th>Dimensión</th><th class="num">% '+per+'</th>'
    + '<th class="num">% '+(idx > 0 ? P[idx-1] : 'anterior')+'</th><th class="num">Variación (pp)</th><th>Estado</th>'
    + '</tr></thead><tbody>'+rows+'</tbody></table></div>';
}

function renderErrores(per){
  const porDim = DIMS.map(d => {
    let det = detalleDim(d.tipo, per);
    if (!det.length){ const f = filaTotalDim(d, per); det = f ? [f] : []; }
    return {label: d.label, color: d.color, val: det.reduce((a, f) => a + f.cantidad_mala, 0), det: det};
  });
  const totalErr = porDim.reduce((a, x) => a + x.val, 0);
  let html = '';
  if (!totalErr){
    html = '<section class="hero"><h2>Análisis de errores</h2>'
      + '<div class="hval">0</div><div class="hlbl">Sin registros con error en '+per+' en las 4 dimensiones.</div></section>';
    document.getElementById('pg-errores').innerHTML = html;
    return;
  }
  const leyenda = porDim.map(x =>
    '<span class="lg"><span class="dot" style="background:'+x.color+'"></span>'+x.label+' · '+fmtInt(x.val)
    + ' ('+(100*x.val/totalErr).toFixed(2)+' %)</span>').join('<br>');
  html += '<div class="grid2"><section><h2>Distribución de errores por dimensión</h2>'
    + '<p class="nota">Total de registros con error en '+per+'.</p>'
    + '<div style="display:flex;gap:20px;align-items:center;flex-wrap:wrap">'
    + donutDist(porDim, totalErr)
    + '<div class="leyenda" style="display:block">'+leyenda+'</div></div></section>';

  const campos = {};
  porDim.forEach(x => x.det.forEach(f => {
    if (f.cantidad_mala > 0) campos[f.nombre_campo] = (campos[f.nombre_campo] || 0) + f.cantidad_mala;
  }));
  const top = Object.entries(campos).sort((a, b) => b[1] - a[1]).slice(0, 10);
  const maxErr = top.length ? top[0][1] : 1;
  const rows = top.map(t =>
    '<tr>'+tdCampo(t[0])
    + '<td><div class="track" data-tip="'+fmtInt(t[1])+' errores"><div class="fill" style="width:'+(100*t[1]/maxErr)+'%"></div></div></td>'
    + '<td class="num">'+fmtInt(t[1])+'</td>'
    + '<td class="num">'+(100*t[1]/totalErr).toFixed(2)+' %</td></tr>').join('');
  html += '<section><h2>Total de errores por campo</h2>'
    + '<p class="nota">Suma de errores del campo en todas las dimensiones (top '+top.length+').</p>'
    + '<div class="twrap"><table><thead><tr><th>Campo</th><th style="width:170px">Relativo al mayor</th>'
    + '<th class="num">Errores</th><th class="num">% del total</th></tr></thead><tbody>'+rows+'</tbody></table></div></section></div>';

  html += '<section style="margin-top:14px"><h2>Pareto — regla 80/20 de campos con mayor volumen de errores</h2>'
    + '<p class="nota">Barras: % del total de errores por campo (la cifra encima es el conteo). '
    + 'Línea: % acumulado. Un solo eje 0–100 %.</p>'
    + pareto(top, totalErr) + '</section>';
  document.getElementById('pg-errores').innerHTML = html;
}

function renderInfo(){
  const u = DATA.umbrales;
  const umbTxt = k => 'BUENO ≥ '+u[k].BUENO+' · ALERTA ≥ '+u[k].ALERTA+' · GRAVE ≥ '+u[k].GRAVE+' · CRÍTICO < '+u[k].GRAVE;
  document.getElementById('pg-info').innerHTML = '<div class="info-grid">'
    + '<section><h3>Nombre del reporte</h3>'
    + '<p class="kv"><b>KPIs Calidad de Datos — CDP Primas</b> (tablero HTML autocontenido)</p>'
    + '<p class="kv"><b>Data Steward:</b> Karen Carvajal - Director de Gobierno de Datos <br><b>Data Owner:</b> Javier Gualdron - Director De Ingenieria De Datos<br>'
    + '<b>Desarrollador del reporte:</b> Wilson Jerez — Analista Senior De Gobierno De Datos<br>'
    + '<b>Esta versión:</b> notebook de solo lectura (SELECT directo, periodos autodetectados)<br>'
    + '<b>Generado:</b> '+DATA.generado+'</p>'
    + '<h3>Descripción del reporte</h3>'
    + '<p>Visión integral de la calidad de datos del modelo CDP Primas, en 4 secciones:</p><ul>'
    + '<li><b>Resumen ejecutivo:</b> KPIs globales por dimensión, DQ Score, tendencia vs. target, radar y semáforo.</li>'
    + '<li><b>Detalle por variable:</b> calidad campo a campo por dimensión, para priorizar remediación.</li>'
    + '<li><b>Tendencia e historia:</b> evolución de los últimos periodos y variación vs. el anterior.</li>'
    + '<li><b>Análisis de errores:</b> distribución por dimensión y campo con Pareto 80/20.</li></ul>'
    + '<h3>Fuentes</h3><ul>'
    + '<li><b>Fuente principal:</b> '+DATA.fuente+'</li>'
    + '<li>Cálculo al vuelo con SELECT (no lee las vistas co_sandbox_datos.vw_kpi_*_mensual_pbi ni el cubo persistido).</li></ul>'
    + '</section>'
    + '<section><h3>Descripción de indicadores</h3><ul>'
    + '<li><b>Completitud:</b> % de campos no nulos sobre el total de registros evaluados.</li>'
    + '<li><b>Validez:</b> % de registros que cumplen reglas de formato, dominio o rango por campo.</li>'
    + '<li><b>Exactitud:</b> % de registros cuyo valor cae en el dominio permitido/esperado por el negocio.</li>'
    + '<li><b>Unicidad:</b> % de registros sin duplicados en la llave (policy_transaction_movement_sk).</li>'
    + '<li><b>DQ Score (Comportamiento general):</b> promedio simple de las 4 dimensiones anteriores.</li>'
    + '<li><b>Disponibilidad:</b> informativa, fuera del DQ Score — sus 2 chequeos (sincronización tabla vs. vista y regla 4 de días 1–15) no son comparables con las demás dimensiones ni entre sí.</li></ul>'
    + '<h3>Umbrales de estado</h3><ul>'
    + '<li>Generales: '+umbTxt('default')+'</li>'
    + '<li>Disponibilidad sincronización: '+umbTxt('disponibilidad_sync')+'</li>'
    + '<li>Disponibilidad regla 4: '+umbTxt('disponibilidad_regla4')+'</li>'
    + '<li>Target de referencia: '+TARGET+' % (línea punteada en tendencia y radar).</li></ul>'
    + '<h3>Notas metodológicas</h3><ul>'
    + '<li>Universo: filtro base compartido (current_record_flag = 1, coverage_code ≠ 8888, prima ≠ 0, iaxis sin unificado-total + as400). Disponibilidad no lo aplica a propósito.</li>'
    + '<li>Exactitud aplica aquí el filtro compartido completo (incluye current_record_flag = 1), a diferencia del SQL original — desviación deliberada documentada (hallazgo #1 en docs/Hallazgos y Estado del Proyecto.md).</li>'
    + '<li>movement_type está excluido de Validez a propósito (también comentado en el SQL original).</li>'
    + '<li>Periodos autodetectados de la fuente: solo meses cerrados, sin fechas quemadas.</li></ul>'
    + '</section></div>';
}

/* ---------- drill-down: clic en un anillo => campos donde falla ---------- */
function abrirDetalle(dim, per){
  let title, body;
  if (dim === 'dq'){
    const rows = DIMS.map(d => {
      const f = filaTotalDim(d, per);
      if (!f) return '';
      const det = detalleDim(d.tipo, per);
      const nf = det.filter(x => x.cantidad_mala > 0).length;
      return '<tr'+(nf ? ' class="falla"' : '')+'><td><span class="dot" style="background:'+d.color+'"></span>'+d.label+'</td>'
        + '<td class="num">'+f.porcentaje.toFixed(2)+'</td>'
        + '<td class="num">'+nf+' de '+det.length+'</td>'
        + '<td class="num">'+fmtInt(det.reduce((a, x) => a + x.cantidad_mala, 0))+'</td>'
        + '<td>'+chip(estado('default', f.porcentaje))+'</td></tr>';
    }).join('');
    title = 'Comportamiento general (DQ Score) — '+per;
    body = '<p class="nota">Promedio simple de las 4 dimensiones comparables. '
      + 'Para el detalle campo a campo de una dimensión, cierre esta ventana y haga clic en su anillo.</p>'
      + '<div class="twrap"><table class="verde"><thead><tr><th>Dimensión</th><th class="num">% Calidad</th>'
      + '<th class="num">Campos con falla</th><th class="num">Registros con error</th><th>Estado</th>'
      + '</tr></thead><tbody>'+rows+'</tbody></table></div>';
    if (REGLAS.dq) body += '<h3>Cómo se calcula</h3><p class="nota">'+escHtml(REGLAS.dq.explicacion)+'</p>';
  } else {
    const d = DIMS.find(x => x.tipo === dim);
    if (!d) return;
    const det = detalleDim(d.tipo, per).slice()
      .sort((a, b) => a.porcentaje - b.porcentaje || b.cantidad_mala - a.cantidad_mala);
    const fallan = det.filter(f => f.cantidad_mala > 0);
    const totErr = fallan.reduce((a, f) => a + f.cantidad_mala, 0);
    title = d.label+' — detalle por campo · '+per;
    const resumen = fallan.length
      ? '<b>'+d.label+' falla en '+fallan.length+' de '+det.length+' campos evaluados</b> ('
        + fmtInt(totErr)+' registros con error). Los campos con falla van resaltados y ordenados de peor a mejor.'
      : '<b>Sin fallas:</b> los '+det.length+' campos evaluados están al 100 % en este periodo.';
    const rows = det.map(f => '<tr'+(f.cantidad_mala > 0 ? ' class="falla"' : '')+'>'
      + tdCampo(f.nombre_campo)
      + '<td class="dcampo">'+(DICC[f.nombre_campo] || '—')+'</td>'
      + '<td class="num">'+f.porcentaje.toFixed(2)+'</td>'
      + '<td class="num">'+fmtInt(f.cantidad_mala)+'</td>'
      + '<td>'+chip(estado('default', f.porcentaje))+'</td></tr>').join('');
    body = '<p class="nota">'+d.desc+'</p><p class="nota">'+resumen+'</p>'
      + (det.length
          ? '<div class="twrap"><table><thead><tr><th>Campo</th><th>Qué es</th><th class="num">%</th>'
            + '<th class="num">'+d.malos+'</th><th>Estado</th></tr></thead><tbody>'+rows+'</tbody></table></div>'
          : '<p class="nota">Sin datos para este periodo.</p>');
    const rg = REGLAS[dim];
    if (rg && rg.reglas){
      const fallaSet = new Set(fallan.map(f => f.nombre_campo));
      const orden = Object.keys(rg.reglas);
      orden.sort((a, b) => (fallaSet.has(b) ? 1 : 0) - (fallaSet.has(a) ? 1 : 0));
      body += '<h3>Cómo se calcula</h3><p class="nota">'+escHtml(rg.explicacion)+'</p>'
        + '<h3>Reglas de cálculo por campo</h3>'
        + '<p class="nota">Explicación de cada regla y el SQL literal con el que se cuentan los «'+d.malos.toLowerCase()+'» '
        + '(cantidad_mala). Los campos con falla aparecen primero y abiertos; el resto, plegado.</p>';
      orden.forEach(campo => {
        const r = rg.reglas[campo];
        const f = det.find(x => x.nombre_campo === campo);
        body += '<details'+(fallaSet.has(campo) ? ' open' : '')+'><summary><span class="campo">'+campo+'</span>'
          + (f ? ' — '+f.porcentaje.toFixed(2)+' % ' : ' ')
          + (f && fallaSet.has(campo) ? chip(estado('default', f.porcentaje), fmtInt(f.cantidad_mala)+' '+d.malos.toLowerCase()) : '')
          + '</summary><p class="nota">'+escHtml(r.que_hace)+'</p><pre class="sql">'+escHtml(r.sql)+'</pre></details>';
      });
      if (rg.universo_sql){
        body += '<details><summary><span class="campo">Universo evaluado</span> — filtro base compartido</summary>'
          + '<p class="nota">'+escHtml(rg.universo || '')+'</p><pre class="sql">'+escHtml(rg.universo_sql)+'</pre></details>';
      }
    }
  }
  document.getElementById('mbox').innerHTML =
    '<div class="mhead"><h2>'+title+'</h2><button class="mclose" title="Cerrar (Esc)">&#10005;</button></div>'+body;
  document.getElementById('modal').classList.add('on');
}
document.getElementById('gauges').addEventListener('click', ev => {
  const g = ev.target.closest('.gauge[data-dim]');
  if (g) abrirDetalle(g.getAttribute('data-dim'), parseInt(document.getElementById('periodo').value, 10));
});
function abrirDetalleDispo(campo, per){
  const dp = DISPO.find(x => x.campo === campo);
  const rg = REGLAS[campo];
  if (!dp || !rg) return;
  const f = fila('disponibilidad', campo, per);
  const body = '<p class="nota">'+dp.desc+'</p>'
    + (f ? '<p class="nota"><b>'+per+':</b> '+f.porcentaje.toFixed(2)+' % · '+fmtInt(f.cantidad_mala)+' de '
         + fmtInt(f.total_registros)
         + (campo === 'disponibilidad_regla4' ? ' ramos con días exigidos faltantes ' : ' registros aún no visibles en la vista ')
         + chip(estado(dp.umbral, f.porcentaje))+'</p>'
       : '<p class="nota">Sin datos en esta corrida.</p>')
    + '<h3>Qué hace la regla</h3><p class="nota">'+escHtml(rg.que_hace)+'</p>'
    + '<h3>Regla de cálculo en SQL</h3><pre class="sql">'+escHtml(rg.sql)+'</pre>';
  document.getElementById('mbox').innerHTML =
    '<div class="mhead"><h2>'+dp.label+' — regla de cálculo</h2><button class="mclose" title="Cerrar (Esc)">&#10005;</button></div>'+body;
  document.getElementById('modal').classList.add('on');
}
document.getElementById('dispo').addEventListener('click', ev => {
  const t = ev.target.closest('.tile[data-dispo]');
  if (t) abrirDetalleDispo(t.getAttribute('data-dispo'), parseInt(document.getElementById('periodo').value, 10));
});
const modal = document.getElementById('modal');
modal.addEventListener('click', ev => {
  if (ev.target === modal || ev.target.closest('.mclose')) modal.classList.remove('on');
});
document.addEventListener('keydown', ev => { if (ev.key === 'Escape') modal.classList.remove('on'); });

/* ---------- pestañas, filtro y arranque ---------- */
const TABS = [
  ['resumen', 'Resumen ejecutivo'], ['detalle', 'Detalle por variable'],
  ['tendencia', 'Tendencia e historia'], ['errores', 'Análisis de errores'], ['info', 'Información'],
];
const nav = document.getElementById('tabs');
nav.innerHTML = TABS.map((t, i) =>
  '<button class="tab'+(i === 0 ? ' on' : '')+'" data-pg="'+t[0]+'">'+t[1]+'</button>').join('');
nav.addEventListener('click', ev => {
  const b = ev.target.closest('.tab');
  if (!b) return;
  nav.querySelectorAll('.tab').forEach(x => x.classList.toggle('on', x === b));
  document.querySelectorAll('.pagina').forEach(p =>
    p.classList.toggle('on', p.id === 'pg-'+b.getAttribute('data-pg')));
});
document.getElementById('pg-resumen').classList.add('on');

const sel = document.getElementById('periodo');
sel.innerHTML = P.slice().reverse().map(p =>
  '<option value="'+p+'">'+p+(p === ENCURSO ? ' (en curso)' : '')+'</option>').join('');
function renderAll(){
  const per = parseInt(sel.value, 10);
  renderResumen(per); renderDetalle(per); renderTendencia(per); renderErrores(per);
}
sel.addEventListener('change', renderAll);

document.getElementById('btnRefresh').addEventListener('click', () => {
  document.getElementById('mbox').innerHTML =
    '<div class="mhead"><h2>Actualizar informe</h2><button class="mclose" title="Cerrar (Esc)">&#10005;</button></div>'
    + '<p class="nota">Este archivo es autocontenido y no puede consultar Redshift por si solo desde el navegador.</p>'
    + '<p class="nota">Para traer datos nuevos:</p>'
    + '<ol style="margin:8px 0 0;padding-left:20px;line-height:1.7">'
    + '<li>Abre la carpeta <code>notebooks/</code> del repo.</li>'
    + '<li>Haz doble clic en <code>actualizar_informe.bat</code> (requiere VPN/red corporativa).</li>'
    + '<li>Se ejecuta el notebook, se regeneran este HTML y <code>reports/cubo_unificado.csv</code>, y el informe se abre solo al terminar.</li>'
    + '</ol>';
  document.getElementById('modal').classList.add('on');
});
document.getElementById('sub').textContent =
  'Actualizado: '+DATA.generado+' · Fuente: '+DATA.fuente+' · Solo lectura'
  + (ENCURSO ? ' · Periodo '+ENCURSO+' EN CURSO: cifras parciales, evolucionan con cada corrida' : '');
document.getElementById('pie').textContent =
  'Tablero autocontenido generado por notebooks/cubo_unificado_runner.ipynb — réplica mejorada del reporte Power BI "Dashboard KPIs Calidad v2".';

/* Tooltip compartido (los valores también están en tablas — el tooltip nunca es la única vía). */
const tip = document.getElementById('tip');
document.addEventListener('mousemove', ev => {
  const t = ev.target.closest ? ev.target.closest('[data-tip]') : null;
  if (t){
    tip.textContent = t.getAttribute('data-tip');
    tip.style.display = 'block';
    tip.style.left = Math.min(ev.clientX + 14, window.innerWidth - tip.offsetWidth - 10)+'px';
    tip.style.top = (ev.clientY + 16)+'px';
  } else tip.style.display = 'none';
});

renderInfo();
renderAll();
</script>
</body>
</html>'''

payload = {
    'generado': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'fuente': 'gde_adp_dwh_vw_general.vw_fact_policy_transaction_movement',
    'periodos': [int(p) for p in PERIODOS],
    'periodo_en_curso': int(PERIODO_EN_CURSO) if PERIODO_EN_CURSO else None,
    'umbrales': UMBRALES,
    'target': TARGET,
    'diccionario': DICCIONARIO_CAMPOS,  # definiciones de campos para el drill-down
    'reglas': REGLAS_DQ,  # explicación + SQL literal de cada regla (modal del drill-down)
    'filas': json.loads(df_cubo.to_json(orient='records')),  # NaN -> null
}

html = HTML_TEMPLATE.replace('__DATA__', json.dumps(payload, ensure_ascii=False))

repo_raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ruta_html = repo_raiz / 'reports' / 'tablero_calidad_primas.html'
ruta_html.parent.mkdir(exist_ok=True)
ruta_html.write_text(html, encoding='utf-8')
print(f'Tablero generado: {ruta_html}')
webbrowser.open(ruta_html.resolve().as_uri())

Tablero generado: C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\tablero_calidad_primas.html


True

## Cerrar conexión

In [22]:
cursor.close()
conn.close()
print('Conexión cerrada.')


Conexión cerrada.
